In [ ]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
import tensorflow_hub as hub
import torch
from transformers import BertTokenizer, BertModel
from google.colab import drive
import pandas as pd
# Ensure necessary NLTK resources are downloaded
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')



# Mount Google Drive
drive.mount('/content/drive')

# Load dataset from Google Drive
file_path = '/content/drive/My Drive/mbti_1.csv'  # Update path as necessary
df = pd.read_csv(file_path)

# Display dataset information
print("Dataset Loaded:")
display(df.head())
display(df.info())

# Text Cleaning and Tokenization
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text)  # Remove mentions and hashtags
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove non-alphanumeric characters
    tokens = word_tokenize(text)  # Tokenization
    return ' '.join(tokens)

df['cleaned_text'] = df['posts'].apply(clean_text)
print("Text Cleaning Completed")

#  Stopword Removal and Lemmatization
stop_words = set(stopwords.words('english'))
lemmatizer = nltk.WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    filtered_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

df['processed_text'] = df['cleaned_text'].apply(preprocess_text)
print("Stopword Removal and Lemmatization Completed")

# Label Encoding Personality Types 0 till 15
label_encoder = LabelEncoder()
df['encoded_labels'] = label_encoder.fit_transform(df['type'])
print("Label Encoding Completed")

# Print the encoding order
print("MBTI Type Encoding Order:")
for i, mbti_type in enumerate(label_encoder.classes_):
    print(f"{i}: {mbti_type}")

# Save LabelEncoder for later use
import joblib
joblib.dump(label_encoder, "label_encoder.pkl")

# Step 4: Handling Class Imbalance using Random Oversampling
X = df['processed_text']
y = df['encoded_labels']
X = np.array(X).reshape(-1, 1)  # Reshaping for oversampling

oversample = RandomOverSampler(sampling_strategy='auto')
X_resampled, y_resampled = oversample.fit_resample(X, y)
X_resampled = X_resampled.flatten()  # Flatten back after oversampling
print("Class Imbalance Addressed with Oversampling")

In [ ]:
import nbformat


nb = nbformat.read(Home_Fit_Final.ipynb, as_version=4)

# remove widget metadata (ROOT FIX)
nb.metadata.pop("widgets", None)

for cell in nb.cells:
    cell.metadata.pop("widgets", None)

output_file = "/content/CLEAN_NOTEBOOK.ipynb"

nbformat.write(nb, output_file)

print("Clean notebook saved:", output_file)

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sklearn.decomposition import PCA
from collections import Counter
import joblib


df = pd.read_csv(file_path)

# Load label encoder
label_encoder = joblib.load("label_encoder.pkl")

# Plot Class Distribution Before Oversampling
plt.figure(figsize=(10, 5))
sns.countplot(x=df['type'], order=df['type'].value_counts().index, palette="viridis")
plt.title("Class Distribution Before Oversampling")
plt.xticks(rotation=45)
plt.show()




# Plot Class Distribution After Oversampling
plt.figure(figsize=(10, 5))
sns.countplot(x=label_encoder.inverse_transform(y_resampled), order=label_encoder.classes_, palette="magma")
plt.title("Class Distribution After Oversampling")
plt.xticks(rotation=45)
plt.show()





In [ ]:
pip install imbalanced-learn


In [ ]:
!pip install --upgrade transformers
!pip install --upgrade tf-keras
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [ ]:
# Splitting dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
print("Data Splitting Completed")
print("X_train size:", len(X_train))
print("X_test size:", len(X_test))
print("y_train size:", len(y_train))
print("y_test size:", len(y_test))


# Words Embedding

In [ ]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print("TF-IDF Vectorization Completed")

print("TF-IDF Vectorization Completed")
print("Shape of X_train_tfidf:", X_train_tfidf.shape)
print("Shape of X_test_tfidf:", X_test_tfidf.shape)
print("\nSample vector (first training instance):")
print(X_train_tfidf[0])


# Glove

In [ ]:
!wget --no-check-certificate https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
!unzip -q glove.6B.zip


In [ ]:
import torch
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import BertTokenizer, BertModel


**BERT Word Embedding**

In [ ]:
import torch
import numpy as np
import os
from transformers import BertTokenizer, BertModel

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def save_embeddings(file_path, new_data):
    """Append new embeddings batch to an existing file or create a new one."""
    if os.path.exists(file_path):
        existing_data = np.load(file_path)
        updated_data = np.vstack([existing_data, new_data])
    else:
        updated_data = new_data
    np.save(file_path, updated_data)

def text_to_bert_embedding_batch(texts, tokenizer, model, batch_size=16, save_path=None):
    all_embeddings = []
    texts = [str(text) for text in texts if isinstance(text, str) and text.strip()]

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        # Tokenization
        inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt', max_length=128)
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Move to GPU

        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling

        batch_embeddings = embeddings.cpu().numpy()
        all_embeddings.append(batch_embeddings)

        # Save after every batch to avoid data loss
        if save_path:
            save_embeddings(save_path, batch_embeddings)

        torch.cuda.empty_cache()  # Free GPU memory

    return np.vstack(all_embeddings)

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()  # Set model to evaluation mode

# Ensure valid input texts
X_train = [str(x) for x in X_train if isinstance(x, str) and x.strip()]
X_test = [str(x) for x in X_test if isinstance(x, str) and x.strip()]

# File paths for incremental saving
train_save_path = "/content/X_train_bert.npy"
test_save_path = "/content/X_test_bert.npy"

# Extract features and save incrementally
X_train_bert = text_to_bert_embedding_batch(X_train, tokenizer, bert_model, batch_size=16, save_path=train_save_path)
X_test_bert = text_to_bert_embedding_batch(X_test, tokenizer, bert_model, batch_size=16, save_path=test_save_path)

print("Feature Extraction Completed")
print("X_train_bert Shape:", X_train_bert.shape)
print("X_test_bert Shape:", X_test_bert.shape)

# Download after saving
from google.colab import files
files.download(train_save_path)
files.download(test_save_path)


In [ ]:
X_train_bert = np.load('/content/X_train_bert.npy')
X_test_bert = np.load('/content/X_test_bert.npy')
# Verify the shape to ensure they are loaded correctly
print("X_train_bert shape:", X_train_bert.shape)
print("X_test_bert shape:", X_test_bert.shape)
print("\nSample BERT vector (first training instance):")
print(X_train_bert[0])

# GloVe

In [ ]:

glove_file_path = '/content/glove.6B.100d.txt'  # Make sure this path matches your extracted location

# Load GloVe embeddings
def load_glove_embeddings(glove_file_path):
    glove_embeddings = {}
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_embeddings[word] = vector
    return glove_embeddings

# Load embeddings
glove_embeddings = load_glove_embeddings(glove_file_path)


print(" GloVe Embeddings Loaded Successfully")
print("Total words loaded:", len(glove_embeddings))

# Check embedding dimension for a known word
sample_word = "modern"
if sample_word in glove_embeddings:
    print(f"Embedding for '{sample_word}': {glove_embeddings[sample_word]}")
    print("Embedding dimension:", glove_embeddings[sample_word].shape[0])
else:
    print(f"'{sample_word}' not found in GloVe embeddings.")

In [ ]:
def text_to_glove_vector(text, glove_embeddings, embedding_dim=100):
    words = text.split()
    word_vectors = [glove_embeddings[word] for word in words if word in glove_embeddings]

    if len(word_vectors) == 0:
        return np.zeros(embedding_dim)  # Return zero vector if no words match GloVe

    return np.mean(word_vectors, axis=0)  # Take mean of word vectors

# Apply GloVe transformation to X_train and X_test
X_train_vectors = np.vstack([text_to_glove_vector(text, glove_embeddings) for text in X_train])
X_test_vectors = np.vstack([text_to_glove_vector(text, glove_embeddings) for text in X_test])

print("Feature Extraction Completed")
print("X_train_vectors Shape:", X_train_vectors.shape)
print("X_test_vectors Shape:", X_test_vectors.shape)

# **Prediction of Personality Type**

In [ ]:
!pip install tabulate catboost xgboost



# Prediction based on BERT Embedding

In [ ]:
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from tabulate import tabulate
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Previous Apporach used for Prediction of Personality type

In [ ]:
# Define classifiers
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVC": SVC(kernel='linear', probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    "SGD Classifier": SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    "CatBoost": CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0)
}

# Store results
results = []
best_model = None
best_accuracy = 0

# Train and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_bert, y_train)
    y_pred = model.predict(X_test_bert)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    # Save the best model
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

    results.append([name, acc, report['weighted avg']['precision'], report['weighted avg']['recall'], report['weighted avg']['f1-score']])

    print(f"{name} Accuracy: {acc:.4f}\n")
    print(classification_report(y_test, y_pred))

# Save best model
joblib.dump(best_model, "/content/best_model.pkl")
print(f"Best Model: {best_model_name} with Accuracy: {best_accuracy:.4f}")


# Stacking Classifier Improvement of Previous Results


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Define base models
estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(eval_metric="mlogloss", use_label_encoder=False)),
    ('svc', SVC(kernel='linear', probability=True))
]

# Define stacking classifier
stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1,
    passthrough=True
)

# Train the stacking model
print("Training Stacking Classifier...")
stack_model.fit(X_train_bert, y_train)

# Predict and evaluate
y_pred = stack_model.predict(X_test_bert)

acc = accuracy_score(y_test, y_pred)
print(f"\n Stacked Accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

# Optionally save the model
import joblib
joblib.dump(stack_model, "/content/stacked_improvedbest_model.pkl")


In [ ]:
# Create table
headers = ["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
print(tabulate(results, headers=headers, tablefmt="grid"))


# Graphical Representation

In [ ]:
# ------------------------ IMPORTS ------------------------
import matplotlib.pyplot as plt
import seaborn as sns

import joblib

# ------------------------ MODELS ------------------------
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVC": SVC(kernel='linear', probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    "SGD Classifier": SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    "CatBoost": CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0)
}

results = []
model_reports = {}
model_predictions = {}
best_model = None
best_accuracy = 0

# ------------------------ TRAIN + EVALUATION ------------------------
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_bert, y_train)
    y_pred = model.predict(X_test_bert)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    results.append([
        name,
        acc,
        report['weighted avg']['precision'],
        report['weighted avg']['recall'],
        report['weighted avg']['f1-score']
    ])

    model_reports[name] = report
    model_predictions[name] = y_pred

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

    print(f"{name} Accuracy: {acc:.4f}\n")

# ------------------------ SAVE BEST MODEL ------------------------
joblib.dump(best_model, "/content/best_model.pkl")
print(f"Best Model: {best_model_name} with Accuracy: {best_accuracy:.4f}")

# ------------------------ RESULTS DF ------------------------
results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])
display(results_df)

# ------------------------ GRAPH 1: MODEL PERFORMANCE COMPARISON ------------------------
plt.figure(figsize=(12,6))
results_df.set_index("Model")[["Accuracy","Precision","Recall","F1"]].plot(kind="bar", figsize=(12,6))
plt.title("Model Comparison (Accuracy / Precision / Recall / F1)")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.ylim(0,1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ------------------------ GRAPH 2: CONFUSION MATRIX PER MODEL ------------------------
labels = sorted(list(set(y_test)))

for name in models.keys():
    cm = confusion_matrix(y_test, model_predictions[name])
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted Personality")
    plt.ylabel("True Personality")
    plt.tight_layout()
    plt.show()

# ------------------------ GRAPH 3: PER-PERSONALITY METRICS PER MODEL ------------------------
for name in models.keys():
    report = model_reports[name]

    # Extract per personality only
    personality_scores = {
        label: {
            "Precision": report[label]["precision"],
            "Recall": report[label]["recall"],
            "F1": report[label]["f1-score"]
        }
        for label in report.keys()
        if label not in ["accuracy", "macro avg", "weighted avg"]
    }

    df_p = pd.DataFrame(personality_scores).T

    plt.figure(figsize=(12,6))
    df_p.plot(kind="bar", figsize=(12,6))
    plt.title(f"Per-Personality Precision / Recall / F1 - {name}")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.ylim(0,1)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


In [ ]:
df_results = pd.DataFrame(results, columns=headers)

# Plot the results
metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
plt.figure(figsize=(12, 6))
df_results.plot(x="Model", y=metrics, kind="bar", colormap="Set2")

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Comparative Analysis


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Create table
headers = ["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
print(tabulate(results, headers=headers, tablefmt="grid"))

# Predict and evaluate stacking model
y_pred = stack_model.predict(X_test_bert)

stacked_results = [["Stacked Classifier",
                    accuracy_score(y_test, y_pred),
                    precision_score(y_test, y_pred, average='weighted', zero_division=0),
                    recall_score(y_test, y_pred, average='weighted', zero_division=0),
                    f1_score(y_test, y_pred, average='weighted', zero_division=0)
                   ]]

print("\nImproved Model (Stacked Classifier) Performance:\n")
print(tabulate(stacked_results, headers=headers, tablefmt="grid"))


# Extract metric names and values
metrics = headers[1:]  # ["Accuracy", "Precision", "Recall", "F1-Score"]
values = stacked_results[0][1:]

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics, values, color='skyblue')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval:.2f}", ha='center', va='bottom')

plt.ylim(0, 1.05)
plt.title("Stacked Classifier Performance")
plt.ylabel("Score")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# Extract metric names and values
metrics = headers[1:]  # ["Accuracy", "Precision", "Recall", "F1-Score"]
values = results[0][1:]

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics, values, color='green')

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval:.2f}", ha='center', va='bottom')

plt.ylim(0, 1.05)
plt.title("ML and Ensemble learner Classifier Performance")
plt.ylabel("Score")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()



In [ ]:
print(best_model.classes_)


# Utilized the Best Model to Predict Personality

**Updated** code for Desire one personality type

In [ ]:
import joblib
import torch
from transformers import BertTokenizer, BertModel

# Load best model
best_model = joblib.load("/content/stacked_improvedbest_model.pkl")

# Load saved LabelEncoder
label_encoder = joblib.load("label_encoder.pkl")

# Load BERT model for embedding
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

def text_to_bert_embedding(text, tokenizer, model):
    """Convert user input text to BERT embedding"""
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.cpu().numpy()

# --- Predict once and save the result ---

user_input = input("Enter your text to predict personality: ")

user_embedding = text_to_bert_embedding(user_input, tokenizer, bert_model)

# Predict personality type (returns an encoded number)
predicted_index = best_model.predict(user_embedding)[0]
display(predicted_index)
# Convert number back to MBTI type
predicted_personality = label_encoder.inverse_transform([predicted_index])[0]

print(f"Predicted Personality Type: {predicted_personality}")

# Now you can use `predicted_personality` in another module
#I enjoy spending time alone reflecting on ideas and exploring theories. I'm very curious and love solving abstract problems, especially ones that involve strategy or creativity. I prefer deep conversations over small talk, and I value independence and authenticity. Sometimes I get lost in thought, but I’m always trying to understand how things connect beneath the surface.


In [ ]:
import joblib
import torch
from transformers import BertTokenizer, BertModel

# Load best model
best_model = joblib.load("/content/best_model.pkl")

# Load saved LabelEncoder
label_encoder = joblib.load("label_encoder.pkl")

# Load BERT model for embedding
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

def text_to_bert_embedding(text, tokenizer, model):
    """Convert user input text to BERT embedding"""
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.cpu().numpy()

# --- Predict once and save the result ---

user_input = input("Enter your text to predict personality: ")

user_embedding = text_to_bert_embedding(user_input, tokenizer, bert_model)

# Predict personality type (returns an encoded number)
predicted_index = best_model.predict(user_embedding)[0]
display(predicted_index)
# Convert number back to MBTI type
predicted_personality = label_encoder.inverse_transform([predicted_index])[0]

print(f"Predicted Personality Type: {predicted_personality}")

# Now you can use `predicted_personality` in another module
#I enjoy spending time alone reflecting on ideas and exploring theories. I'm very curious and love solving abstract problems, especially ones that involve strategy or creativity. I prefer deep conversations over small talk, and I value independence and authenticity. Sometimes I get lost in thought, but I’m always trying to understand how things connect beneath the surface.


Saved in file(for Explainability)

In [ ]:
print(f"Predicted Personality Type: {predicted_personality}")

# Save predicted personality to a file
with open("predicted_personality.txt", "w") as f: f.write(predicted_personality)


Testing Purpose (Predict more personalities)

In [ ]:
# ---- TAKE 3 NEW TEST INPUTS WITHOUT RERUNNING ANY MODEL LOADING ----

def predict_personality_only(text):
    emb = text_to_bert_embedding(text, tokenizer, bert_model)
    idx = best_model.predict(emb)[0]
    pers = label_encoder.inverse_transform([idx])[0]
    return {"text": text, "embedding": emb, "encoded": idx, "personality": pers}

print("\nEnter Test Input 1:")
text1 = input()
test_p1 = predict_personality_only(text1)

print("\nEnter Test Input 2:")
text2 = input()
test_p2 = predict_personality_only(text2)

print("\nEnter Test Input 3:")
text3 = input()
test_p3 = predict_personality_only(text3)

# ---- SHOW RESULTS ----
print("\n===== STORED PERSONALITY TEST RESULTS =====")
print("Test_P1 Personality:", test_p1["personality"])
print("Test_P2 Personality:", test_p2["personality"])
print("Test_P3 Personality:", test_p3["personality"])

# ---- SAVE TO FILE ----
with open("testing_per_type_personality.txt", "w") as f:
    f.write("Test_P1: " + test_p1["personality"] + "\n")
    f.write("Test_P2: " + test_p2["personality"] + "\n")
    f.write("Test_P3: " + test_p3["personality"] + "\n")

print("\nSaved all predicted types to: testing_per_type.txt")


# Explainability


In [ ]:
!pip install shap
!pip install captum


# SHAP (SHapley Additive Explanations): to determine the most influential dimensions in the BERT embeddings.

# Integrated Gradients (IG): to provide token-level insight into the model's prediction logic.

# Explainability for predicted type using SHAP

In [ ]:
import joblib
import shap
import numpy as np
import torch
from transformers import BertTokenizer, BertModel

# Load model and encoders
best_model = joblib.load("best_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")

# BERT setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

def text_to_bert_embedding(text):
    """Convert user input to BERT embedding"""
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)

    return embedding.cpu().numpy()

# ---- Step 1: Predict personality type ----

user_embedding = text_to_bert_embedding(user_input)
predicted_index = best_model.predict(user_embedding)[0]
predicted_type = label_encoder.inverse_transform([predicted_index])[0]

print(f"\n✅ Predicted Personality Type: {predicted_type} (index {predicted_index})")

# ---- Step 2: SHAP Explainability ----
explainer = shap.TreeExplainer(best_model)
shap_values_all = explainer.shap_values(user_embedding)

# If shap_values_all is a list → multi-class, get the one for predicted class
if isinstance(shap_values_all, list) and len(shap_values_all) > 1:
    shap_values = shap_values_all[predicted_index]
    base_value = explainer.expected_value[predicted_index]
else:
    # fallback (e.g. binary model)
    shap_values = shap_values_all[0]
    base_value = explainer.expected_value

# ---- Step 3: Construct SHAP Explanation object ----
feature_names = [f"Embedding_{i}" for i in range(user_embedding.shape[1])]

shap_exp = shap.Explanation(
    values=shap_values[0],
    base_values=base_value,
    data=user_embedding[0],
    feature_names=feature_names
)

# ---- Step 4: Plot SHAP explanation ----
shap.plots.bar(shap_exp, max_display=30)  # adjust max_display to control visible features


# Word-level SHAP Explanation for BERT Classifier

In [ ]:
import joblib
import shap
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from transformers import BertForSequenceClassification

# Load your model and tokenizer
best_model = joblib.load("best_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Device for torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load raw BERT model for embedding
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device)
bert_model.eval()

# ✨ SHAP requires a wrapper that takes raw text and returns model prediction probabilities
def predict_proba(texts):
    # Convert text to BERT embeddings
    embeddings = []
    for text in texts:
        inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
            pooled = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.append(pooled[0])
    embeddings = np.array(embeddings)
    return best_model.predict_proba(embeddings)


# Predict the type
embedding = predict_proba([user_input])
predicted_index = np.argmax(embedding)
predicted_personality = label_encoder.inverse_transform([predicted_index])[0]
print(f"\n✅ Predicted MBTI Type: {predicted_personality}")

# ---- SHAP Explainability (word-level) ----
explainer = shap.Explainer(predict_proba, tokenizer)
shap_values = explainer([user_input])

# 🔍 Visualize: Which words influence prediction of MBTI type
shap.plots.text(shap_values[0])


# top 20 influential words (positive or negative)

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="bert-base-uncased", return_all_scores=True)

def predict_prob(texts):
    preds = []
    for t in texts:
        out = classifier(t)[0]
        # convert to np array of probabilities in label order
        probs = [x['score'] for x in sorted(out, key=lambda x: x['label'])]
        preds.append(probs)
    return np.array(preds)

explainer = shap.Explainer(predict_prob, masker=shap.maskers.Text(tokenizer))
shap_values = explainer([user_input])
shap.plots.text(shap_values[0])


In [ ]:
import shap

# Use a masker for raw text
masker = shap.maskers.Text(tokenizer)

# Wrapper for sklearn model to take raw text -> embedding -> predict_proba
def predict_raw_text(texts):
    embeddings = []
    for t in texts:
        emb = text_to_bert_embedding(t)
        embeddings.append(emb[0])
    embeddings = np.array(embeddings)
    return best_model.predict_proba(embeddings)

# Create explainer
explainer = shap.Explainer(predict_raw_text, masker)

# Explain
shap_values = explainer([user_input])

# Display words that mattered the most
shap.plots.text(shap_values[0])


In [ ]:
import pandas as pd
import shap
import numpy as np

# ---- Use the same masker explainer as before ----
masker = shap.maskers.Text(tokenizer)

def predict_raw_text(texts):
    embeddings = []
    for t in texts:
        emb = text_to_bert_embedding(t)  # your existing function
        embeddings.append(emb[0])
    embeddings = np.array(embeddings)
    return best_model.predict_proba(embeddings)

explainer = shap.Explainer(predict_raw_text, masker)
shap_values = explainer([user_input])

# ---- Force personality to INFP ----
predicted_personality = "INFP"
pred_class_idx = np.where(label_encoder.classes_ == predicted_personality)[0][0]

# ---- Extract tokens and their SHAP values ----
tokens = shap_values.data[0]          # the words/tokens in the input
importances = shap_values.values[0][:, pred_class_idx]  # SHAP contribution for INFP

# ---- Build DataFrame ----
df_words = pd.DataFrame({
    "Predicted_Personality": [predicted_personality]*len(tokens),
    "Word": tokens,
    "Impact": importances,
    "Direction": ["supports" if v > 0 else "opposes" for v in importances]
})

# ---- Sort by absolute impact ----
df_words = df_words.reindex(df_words["Impact"].abs().sort_values(ascending=False).index)

# ---- Display and save ----
print(f"\nWords contributing most to predicted personality ({predicted_personality}):")
print(df_words)

df_words.to_csv(f"{predicted_personality}_word_shap.csv", index=False)
print(f"\n✅ Saved all word contributions to '{predicted_personality}_word_shap.csv'")


In [ ]:
import pandas as pd

# ---- Read the top 20 words file ----
predicted_personality = "INFP"
file_name = f"{predicted_personality}_top20_word_shap.csv"

df_top20 = pd.read_csv("/content/INFP_word_shap.csv")

# ---- Display the table ----
print(f"\nTop 20 words contributing to predicted personality ({predicted_personality}):")
display(df_top20)  # nicely formatted table in Jupyter/Colab


# Prediction based on GloVe

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVC": SVC(kernel='linear', probability=True),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="mlogloss"),
    "SGD Classifier": SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    "CatBoost": CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, verbose=0)
}

# Evaluate models
results = []
best_model = None
best_accuracy = 0

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_vectors, y_train)
    y_pred = model.predict(X_test_vectors)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    results.append([
        name,
        round(acc, 4),
        round(report["weighted avg"]["precision"], 4),
        round(report["weighted avg"]["recall"], 4),
        round(report["weighted avg"]["f1-score"], 4)
    ])

    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

    joblib.dump(model, f"{name.replace(' ', '_').lower()}_model.pkl")

# Save best model
joblib.dump(best_model, "best_model.pkl")

# Display results in a clean table format
headers = ["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
print("\nEvaluation Report:\n")
print(tabulate(results, headers=headers, tablefmt="grid"))

# Plot Accuracy Comparison
results_df = pd.DataFrame(results, columns=headers)
plt.figure(figsize=(10, 6))
bars = plt.bar(results_df["Model"], results_df["Accuracy"], color='teal')
plt.title("Model Accuracy Comparison (GloVe)", fontsize=14)
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f"{height:.2f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.grid(axis='y')
plt.show()


# Interior Design Dataset Part Upgraded version With ROS and without ROS

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
!pip install pyspellchecker
from spellchecker import SpellChecker
!pip install sentence-transformers faiss-cpu numpy --quiet
from sentence_transformers import SentenceTransformer
import faiss
import pickle

In [ ]:
drive.mount('/content/drive')

file_path_interior = '/content/drive/My Drive/dataset_extracted_all_record.csv'
df_interior_db_ = pd.read_csv(file_path_interior)

df_with_ROS=df_interior_db_.copy()

# Display first few rows
print("First 5 rows of the dataset:")
display(df_interior_db_)

# Show dataset shape
print(f"\nDataset contains {df_interior_db_.shape[0]} rows and {df_interior_db_.shape[1]} columns.")

# Show column names
print("\nColumns in the dataset:")
print(df_interior_db_.columns.tolist())

# Show dataset info
print("\nDataset Info:")
df_interior_db_.info()

# Show basic statistics
print("\nDataset Statistics:")
display(df_interior_db_.describe(include="all"))

# Data Preprocessing

In [ ]:


# Initialize spell checker
spell = SpellChecker()
def clean_description(text):
    # Lowercase
    text = text.lower()

    # Remove non-letter characters (keep spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Split into words
    words = text.split()

    # Remove repeated words (while preserving order)
    seen = set()
    unique_words = []
    for word in words:
        if word not in seen:
            seen.add(word)
            unique_words.append(word)

    # Correct spelling
    corrected = [spell.correction(word) if word not in spell else word for word in unique_words]

    return ' '.join(corrected)


In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
import tensorflow_hub as hub
import torch
from transformers import BertTokenizer, BertModel
from google.colab import drive
# Ensure necessary NLTK resources are downloaded
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

nltk.download('punkt')
nltk.download('stopwords')
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()  # Convert to lowercase
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove punctuation
        tokens = word_tokenize(text)  # Tokenization
        tokens = [word for word in tokens if word not in stopwords.words('english')]  # Remove stopwords
        return ' '.join(tokens)
    return text

df_interior_db_['description'] = df_interior_db_['description'].apply(clean_text)


In [ ]:
# Apply cleaning to description column
df_interior_db_['cleaned_description'] = df_interior_db_['description'].astype(str).apply(clean_description)

df_with_ROS['cleaned_description'] = df_with_ROS['description'].astype(str).apply(clean_description)

# Preview cleaned version
display(df_interior_db_[['description', 'cleaned_description']].head(5))
display(df_with_ROS[['description', 'cleaned_description']].head(5))


# Random Over Sampling

In [ ]:
# Class distribution before balancing
plt.figure(figsize=(10, 5))
sns.countplot(x=df_with_ROS['predicted_style'], palette='viridis')
plt.title("Class Distribution Before Balancing")
plt.xlabel("Predicted Style")
plt.ylabel("Count")
plt.show()

# Handle Class Imbalance in 'predicted_style' Using Random Over-Sampling (ROS)
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(df_with_ROS.drop(columns=['predicted_style']), df_with_ROS['predicted_style'])
df_with_ROS = pd.concat([X_resampled, y_resampled], axis=1)

# Class distribution after balancing
plt.figure(figsize=(10, 5))
sns.countplot(x=df_with_ROS['predicted_style'], palette='viridis')  # Updated to use df_interior_db
plt.title("Class Distribution After Balancing")
plt.xlabel("Predicted Style")
plt.ylabel("Count")
plt.show()


print("Class imbalance handled and descriptions cleaned.")

In [ ]:
print("Dataset Without ROS")

df_interior_db_.shape
df_interior_db_.describe()

In [ ]:

print("Dataset With ROS")
df_with_ROS.shape
df_with_ROS.describe()


# Predicted personality mapping code without Random Over Sampling

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
# --- Assume 'predicted_personality' is already available from previous module ---

# Your strict style rules
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_correlation_matrix():
    """Create personality correlation matrix based on style rules."""
    df_rules = pd.DataFrame([
        {'personality_type': pt, 'styles': styles}
        for pt, styles in STYLE_RULES.items()
    ])

    mlb = MultiLabelBinarizer()
    style_encoded = mlb.fit_transform(df_rules['styles'])

    corr_matrix = pd.DataFrame(
        cosine_similarity(style_encoded),
        columns=df_rules['personality_type'],
        index=df_rules['personality_type']
    )

    return corr_matrix

def fetch_matching_designs(personality_type, df_designs):
    """Fetch designs matching the personality type's styles."""
    if personality_type not in STYLE_RULES:
        raise ValueError(f"Invalid personality type. Must be one of: {list(STYLE_RULES.keys())}")

    target_styles = STYLE_RULES[personality_type]

    mask = df_designs['predicted_style'].apply(
        lambda x: any(style.lower() in x.lower() for style in target_styles)
    )

    return df_designs[mask].copy()


if __name__ == "__main__":


    user_personality = predicted_personality

    corr_matrix = create_correlation_matrix()
    print("Personality Style Correlation Matrix:")
    print(corr_matrix)

    try:
        print(f"\nMost similar personalities to {user_personality}:")
        similar = corr_matrix[user_personality].sort_values(ascending=False)[1:4]
        print(similar.to_string())

        matching_designs = fetch_matching_designs(user_personality, df_interior_db_)

        matching_designs["personality_type"] = user_personality

        print(f"\nFound {len(matching_designs)} matching designs:")
        print("="*80)
        print(matching_designs[['folder_name', 'image_name', 'predicted_style', 'description', 'cleaned_description', 'personality_type']].head())
        print("="*80)

        matching_designs.to_csv(f"{user_personality}_designs.csv", index=False)

    except ValueError as e:
        print(f"Error: {e}")


# Mapping with Random Over Sampling Balanced Dataset

In [ ]:

# --- Assume 'predicted_personality' is already available from previous module ---

# Your strict style rules
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_correlation_matrix():
    """Create personality correlation matrix based on style rules."""
    df_rules = pd.DataFrame([
        {'personality_type': pt, 'styles': styles}
        for pt, styles in STYLE_RULES.items()
    ])

    mlb = MultiLabelBinarizer()
    style_encoded = mlb.fit_transform(df_rules['styles'])

    corr_matrix = pd.DataFrame(
        cosine_similarity(style_encoded),
        columns=df_rules['personality_type'],
        index=df_rules['personality_type']
    )

    return corr_matrix

def fetch_matching_designs(personality_type, df_designs):
    """Fetch designs matching the personality type's styles."""
    if personality_type not in STYLE_RULES:
        raise ValueError(f"Invalid personality type. Must be one of: {list(STYLE_RULES.keys())}")

    target_styles = STYLE_RULES[personality_type]

    mask = df_designs['predicted_style'].apply(
        lambda x: any(style.lower() in x.lower() for style in target_styles)
    )

    return df_designs[mask].copy()

# Example usage
if __name__ == "__main__":
    # Assuming df_with_ROS is already loaded with your columns:
    # folder_name, image_name, dominant_color, palette, description, predicted_style

    # --- Use the obtained predicted_personality ---
    user_personality = predicted_personality

    # 1. Create correlation matrix
    corr_matrix = create_correlation_matrix()
    print("Personality Style Correlation Matrix:")
    print(corr_matrix)

    try:
        # 2. Show similar personalities
        print(f"\nMost similar personalities to {user_personality}:")
        similar = corr_matrix[user_personality].sort_values(ascending=False)[1:4]
        print(similar.to_string())

        # 3. Fetch matching designs
        matching_designs = fetch_matching_designs(user_personality, df_with_ROS)

        # ✅ Add personality type to each row
        matching_designs["personality_type"] = user_personality

        print(f"\nFound {len(matching_designs)} matching designs:")
        print("="*80)
        print(matching_designs[['folder_name', 'image_name', 'predicted_style', 'description', 'cleaned_description', 'personality_type']].head())
        print("="*80)

        # ✅ Save to CSV with type info
        matching_designs.to_csv(f"{user_personality}_designs_ROS.csv", index=False)

    except ValueError as e:
        print(f"Error: {e}")


# Association Rules based on Literature Review Reference: **Huang, Z. Adaptive interior design method for different MBTI personality types based on generative artificial intelligence. ARIN 3, 23 (2024). https://doi.org/10.1007/s44223-024-00066-z**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# --- Strict Style Rules ---
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

def create_full_style_heatmap(df_interior_db):
    """
    Create and display a heatmap showing the correlation
    between MBTI personality types and styles from the database.
    """
    try:
        # --- 1. Prepare Data ---
        all_db_styles = sorted(df_interior_db['predicted_style'].str.lower().unique())
        personality_types = sorted(STYLE_RULES.keys())

        # --- 2. Initialize Correlation Matrix ---
        corr_matrix = pd.DataFrame(
            index=personality_types,
            columns=all_db_styles,
            data=0.0
        )

        # --- 3. Fill Correlation Values ---
        for personality in personality_types:
            preferred_styles = [style.lower() for style in STYLE_RULES[personality]]
            for db_style in all_db_styles:
                if db_style in preferred_styles:
                    corr_matrix.loc[personality, db_style] = 1.0
                elif any(db_style in pref or pref in db_style for pref in preferred_styles):
                    corr_matrix.loc[personality, db_style] = 0.5

        # --- 4. Plot Heatmap ---
        plt.figure(figsize=(18, 10))
        cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#ffffff", "#ffe066", "#228B22"])

        sns.heatmap(
            corr_matrix,
            annot=True,
            fmt=".1f",
            cmap=cmap,
            vmin=0,
            vmax=1,
            linewidths=0.6,
            linecolor='gray',
            cbar_kws={'label': 'Match Strength'},
            annot_kws={"size": 8}
        )

        plt.title(
            "MBTI Personality Types vs Interior Design Styles\n"
            "Green = Strong Match | Yellow = Partial Match | White = No Match",
            fontsize=16,
            pad=20
        )
        plt.xlabel("Interior Design Styles", fontsize=14)
        plt.ylabel("Personality Types", fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(fontsize=10)
        plt.tight_layout()
        plt.show()

        return corr_matrix

    except Exception as e:
        print(f"Error creating heatmap: {e}")
        return None

if __name__ == "__main__":
    correlation_data = create_full_style_heatmap(df_interior_db_)

    if correlation_data is not None:
        print("\nSummary of Correlation Matrix:")
        print(correlation_data.round(2))


# Applied RAG on Random over sampling and included Description

In [ ]:

import pandas as pd
df_type = pd.read_csv(f"{user_personality}_designs_ROS.csv")



print("Columns in loaded CSV:", df_type.columns.tolist())


# --- Construct context using all relevant features ---
df_type["design_context_without_desc"] = df_type.apply(
    lambda row: f"""
    Personality Type: {row['personality_type']}
    Design Details:
    - Dominant Color: {row['dominant_color']}
    - Color Palette: {row['palette']}
    - style: {row['predicted_style']}
    - description: {row['cleaned_description']}

    """,
    axis=1
)

# --- Convert context column to list for downstream processing ---
designs = df_type["design_context_without_desc"].tolist()

# --- Display first few design contexts ---
print("\n✅ Sample design contexts:")
for d in designs[:3]:
    print(d)
    print("=" * 50)

# Preprocessing the dataframe  

In [ ]:
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
import numpy as np

# Create a KDTree for fast nearest neighbor search
css3_db = mcolors.CSS4_COLORS  # You can also use CSS4_COLORS for a richer set
names = list(css3_db.keys())
rgb_values = [mcolors.to_rgb(css3_db[name]) for name in names]
kdtree = KDTree(rgb_values)

# Parse the palette string into a list of RGB tuples
def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

# Convert RGB tuple to nearest color name
def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

# Apply processing to the DataFrame
df_type['palette_names'] = df_type['palette'].apply(
    lambda x: ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

# Show example output
print(df_type[['palette', 'palette_names']].head())


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import faiss

# Initialize model
model = SentenceTransformer('paraphrase-MiniLM-L12-v2')

# Combine all relevant features into one embedding string
embedding_inputs = [
    f"Personality: {row['personality_type']} | Color: {row['dominant_color']} | Style: {row['predicted_style']} | Palette: {row['palette_names']} | description: {row['cleaned_description']}"
    for _, row in df_type.iterrows()
]

display(embedding_inputs)
# Generate embeddings
embeddings = model.encode(embedding_inputs)
embeddings = np.array(embeddings).astype('float32')

# Save embeddings and context
with open('design_embeddings_with_ROS.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': embeddings,
        'design_contexts': embedding_inputs  # full string input used
    }, f)

# Save FAISS index
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
faiss.write_index(faiss_index, 'design_index_with_ROS.faiss')

print("✅ Embeddings generated and saved with personality type included.")


In [ ]:
# Load embeddings
import pickle
import faiss

with open('design_embeddings_with_ROS.pkl', 'rb') as f:
    data = pickle.load(f)
    embeddings = data['embeddings']
    designs = data['design_contexts']

# Load FAISS index
index = faiss.read_index('design_index_with_ROS.faiss')

# Verify loading
print(f"Loaded {len(designs)} designs with {embeddings.shape[1]}-dim embeddings")

In [ ]:
# Create FAISS index and add embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Save index
faiss.write_index(index, 'design_index_with_ROS.faiss')


In [ ]:
def retrieve_designs(query, k=3):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    retrieved_contexts = "\n\n".join([designs[i] for i in indices[0]])
    return retrieved_contexts, query_embedding

# Prompt Engineering

# RAG framework with instruction based prompting, task decomposition, and soft expert role prompting to form a structured prompt engineering approach (INFP )

In [ ]:
def generate_design_prompts(k=3):
    prompts = []

    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)  # Get the embedding of the current design context
        distances, indices = faiss_index.search(query_embedding, k + 1)  # Search for the most similar designs
        similar_indices = [idx for idx in indices[0] if idx != i][:k]  # Exclude self and take the top k similar indices

        similar_contexts = [designs[j] for j in similar_indices]  # Get similar contexts based on the indices

        # Refined and concise prompt for better relevance and creativity
        prompt = f"""You are a visionary interior designer with a deep understanding of personality-driven design. Your task is to create a space that not only represents the user’s personality but also nurtures their emotional needs.

**Primary Design Inspiration:**
{context}

**Related Inspirations:**
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompt += f"""
Design this space thoughtfully, considering the user’s emotional connection to the environment:
- Choose a **primary color** that captures the user’s mood and enhances their personality.
- Select **furniture** that aligns with their emotional needs, comfort, and expression.
- Pick **lighting** that either energizes or calms, depending on their preference.

Finally, describe the **overall emotional vibe** of the space and explain how the elements of design (color, furniture, and lighting) seamlessly come together to create a cohesive and emotionally fulfilling environment.
"""

        prompts.append(prompt)

    return prompts

# --- Now generate the design prompts ---
design_prompts = generate_design_prompts(k=3)

# --- Preview the first few prompts ---
for p in design_prompts[:1]:
    print(p)
    print("=" * 80)


Updated prompt code to fetch top k ref

In [ ]:
def generate_design_prompts(k=3):
    prompts = []
    top_k_references_all = []  # store top-k references per prompt

    for i, context in enumerate(designs):
        query_embedding = np.expand_dims(embeddings[i], axis=0)
        distances, indices = faiss_index.search(query_embedding, k + 1)
        similar_indices = [idx for idx in indices[0] if idx != i][:k]

        similar_contexts = [designs[j] for j in similar_indices]

        prompt = f"""You are a visionary interior designer with a deep understanding of personality-driven design.
        Your task is to create a space that not only represents the user’s personality but also nurtures their emotional needs.

**Primary Design Inspiration:**
{context}

**Related Inspirations:**
""" + "\n".join([f"{j+1}. {sc}" for j, sc in enumerate(similar_contexts)])

        prompt += f"""
Design this space thoughtfully, considering the user’s emotional connection to the environment:
- Choose a **primary color** that captures the user’s mood and enhances their personality.
- Select **furniture** that aligns with their emotional needs, comfort, and expression.
- Pick **lighting** that either energizes or calms, depending on their preference.

Finally, describe the **overall emotional vibe** of the space and explain how the elements of design (color, furniture, and lighting) seamlessly come together to create a cohesive and emotionally fulfilling environment.
"""

        prompts.append(prompt)
        top_k_references_all.append(similar_contexts)  # save top-k references

    return prompts, top_k_references_all

design_prompts, top_k_references_all = generate_design_prompts(k=3)

# Preview
for p, refs in zip(design_prompts[:1], top_k_references_all[:1]):
    print("Prompt:\n", p)
    print("\nTop-k References:\n", refs)
    print("="*80)



# Generate Recommendations using LLAMA3.3_70B_8192

In [ ]:
import requests
import time

api_key="MY_API_KEY";
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}
#changed the api and version from llama3 70 b to llama 3.3 70b versatile
def generate_llama3_recommendations(prompts, model="llama-3.3-70b-versatile", temperature=0.5):
    responses = []
    for i, prompt in enumerate(prompts):
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a brilliant, emotionally intuitive interior designer. Make your recommendations creative,concise, expressive,complete and glitterified."},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature,
            "max_tokens": 768
        }

        try:
            res = requests.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload)

            # Check for errors
            if res.status_code != 200:
                print(f"[❌] Prompt {i+1} failed: HTTP {res.status_code} → {res.text}")
                responses.append("")
                continue

            data = res.json()
            output = data["choices"][0]["message"]["content"]
            print(f"[✅] Prompt {i+1} completed.")
            responses.append(output)

        except requests.exceptions.RequestException as e:
            print(f"[❌] Network error for prompt {i+1}: {e}")
            responses.append("")
        except Exception as e:
            print(f"[❌] Unexpected error for prompt {i+1}: {e}")
            responses.append("")

        time.sleep(0.5)  # avoid rate limits
    return responses



In [ ]:
# Assuming you've already created semantic_prompts
llama3_outputs = generate_llama3_recommendations(design_prompts[:1])  # Do first
display(llama3_outputs)

# **Evaluation matric for generated recommendations**


In [ ]:
!pip install rouge_score

# Reference-Free Semantic Evaluation of LLM-Generated Recommendations
Reason :BLEU and ROUGE require gold references and fail for creative or descriptive outputs.
Since we do not have expected outputs, using input prompts as references gives
misleadingly low scores. Instead, we use:
**Semantic P-BLEU & ROUGE-L: embedding-based metrics for paraphrased n-gram and LCS similarity.**

In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import numpy as np
from nltk.util import ngrams

# --- Load SBERT ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic P-BLEU (reference-free, input-based) ---
def semantic_p_bleu(hypo_text, topk_refs, n=3, sim_threshold=0.3):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = model_sbert.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in topk_refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = model_sbert.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L (reference-free) ---
def semantic_rouge_l(hypo_text, topk_refs, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = model_sbert.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in topk_refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = model_sbert.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluation ---
results = []

for i, hypo_text in enumerate(llama3_outputs):
    topk_refs = top_k_references_all[i]  # exact 3 similar inputs used in prompt

    p_bleu = semantic_p_bleu(hypo_text, topk_refs)
    rouge_l = semantic_rouge_l(hypo_text, topk_refs)

    # Compute BERTScore separately

    results.append({
        "P-BLEU": round(p_bleu,3),
        "ROUGE-L": round(rouge_l,3),
    })

df = pd.DataFrame(results)
display(df)


In [ ]:
!pip install bert-score


#BERT and SBERT Score(Semantic evaluation)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# --- Load SBERT model ---
model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

# --- Initialize results list ---
results = []

# Loop through all generated recommendations
for i in range(len(llama3_outputs)):
    generated = llama3_outputs[i]
    refs = top_k_references_all[i]  # list of top-k reference strings for this prompt

    # --- BERTScore (precision, recall, F1) ---
    # Pick best reference based on max SBERT similarity for BERTScore
    best_ref = max(refs, key=lambda r: util.cos_sim(
        model_sbert.encode(r, convert_to_tensor=True),
        model_sbert.encode(generated, convert_to_tensor=True)
    ).item())
    P, R, F1 = bert_score([generated], [best_ref], lang="en")
    P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

    # --- SBERT semantic alignment (max cosine similarity among top-k refs) ---
    emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
    emb_gen = model_sbert.encode(generated, convert_to_tensor=True)
    cos_scores = util.cos_sim(emb_gen, emb_refs)
    sbert_alignment = cos_scores.max().item()

    # --- Save results ---
    results.append({
        "BERTScore_P": round(P, 3),
        "BERTScore_R": round(R, 3),
        "BERTScore_F1": round(F1, 3),
        "SBERT_Alignment": round(sbert_alignment, 3)
    })

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)

# ==========================================================
# --- Graphical Representation ---
# ==========================================================
plt.figure(figsize=(12,6))
plt.plot(df_metrics.index, df_metrics["BERTScore_F1"], marker='o', label="BERTScore F1")
plt.plot(df_metrics.index, df_metrics["SBERT_Alignment"], marker='s', label="SBERT Alignment (Cosine)")
plt.title("Evaluation of Generated Recommendations")
plt.xlabel("Prompt Index")
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.grid(True)
plt.legend()
plt.show()

# Optional: side-by-side bar chart
df_metrics[["BERTScore_F1","SBERT_Alignment"]].plot(
    kind="bar", figsize=(12,6), rot=0, title="BERTScore vs SBERT Alignment"
)
plt.ylabel("Score")
plt.show()


# Evalution of Model  

# Generate Recommendations using GPT-4

In [ ]:
pip install openai


In [ ]:
from openai import OpenAI
import time

# Initialize OpenAI client
client = OpenAI(api_key="MY_OPENAI_API_KEY")
ans = client.models.list()

for model in ans.data:
    print(model.id)
# Function to get GPT responses for your auto-generated prompts
def generate_responses_from_prompts(prompts, model="gpt-4o-mini", temperature=0.7, max_tokens=200):
    responses = []

    for i, prompt in enumerate(prompts):
        print(f"Sending prompt {i+1}/{len(prompts)}...")

        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a helpful interior design assistant."},
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            response_text = completion.choices[0].message.content.strip()
            responses.append((prompt, response_text))
        except Exception as e:
            print(f"Error with prompt {i}: {e}")
            responses.append((prompt, "ERROR"))

        time.sleep(1)  # Avoid rate limits

    return responses


In [ ]:
# Use your generated prompts (e.g., from generate_automatic_prompts())
test_prompts = design_prompts[:1] # Only send first 5 to avoid cost/rate issues

results = generate_responses_from_prompts(test_prompts)

# Print results
for i, (prompt, response) in enumerate(results):
    print(f"\n--- PROMPT {i+1} ---\n{prompt}")
    print(f"\n--- GPT-4o-mini RESPONSE ---\n{response}")
    print("="*80)


# Evalution of GPT

In [ ]:
#evalution of 19 sep
import pandas as pd
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
!pip install bert-score

from bert_score import score as bert_score

# --- Functions ---

# BLEU Score (with smoothing method4)
def bleu_score(reference, hypothesis):
    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference.split()], hypothesis.split(), smoothing_function=smoothie)

# BERTScore (Precision, Recall, F1)
def bert_scores(reference, hypothesis, lang="en"):
    P, R, F1 = bert_score([hypothesis], [reference], lang=lang)
    return float(P[0]), float(R[0]), float(F1[0])

# --- Generate Responses ---
test_prompts = design_prompts[:1]   # test with 1 (or [:5] for 5 prompts)
results = generate_responses_from_prompts(test_prompts)

# --- Evaluation Loop ---
eval_rows = []

for i, (prompt, response) in enumerate(results):
    bleu = bleu_score(prompt, response)
    bert_p, bert_r, bert_f1 = bert_scores(prompt, response)

    row = {
        "Prompt": prompt[:50].strip().replace("\n", " ") + "...",
        "Response": response[:100].strip().replace("\n", " ") + "...",
        "BLEU (method4)": round(bleu, 3),
        "BERTScore (Precision)": round(bert_p, 3),
        "BERTScore (Recall)": round(bert_r, 3),
        "BERTScore (F1)": round(bert_f1, 3),
    }
    eval_rows.append(row)

df = pd.DataFrame(eval_rows)
display(df)

# --- Visualization ---
plt.figure(figsize=(10,6))
plt.plot(df.index, df["BLEU (method4)"], marker='o', label="BLEU (method4)")
plt.plot(df.index, df["BERTScore (F1)"], marker='s', label="BERTScore (F1)")

plt.title("Evaluation of GPT Responses (BLEU vs BERTScore)")
plt.xlabel("Prompt Index")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()

# Optional: Bar chart
df[["BLEU (method4)", "BERTScore (F1)"]].plot(
    kind="bar", figsize=(10,6), rot=0, title="BLEU vs BERTScore(F1) by Prompt"
)
plt.ylabel("Score")
plt.show()


# LoRa Fine Tuning Final Version

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton nltk sentence-transformers bert-score rouge-score

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
import pandas as pd, yaml
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = torch.bfloat16,   # ← explicitly set
    load_in_4bit = load_in_4bit,
)



model = FastLanguageModel.get_peft_model(
model,
r=16,
target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
lora_alpha=16,
lora_dropout=0,
bias="none",
use_gradient_checkpointing="unsloth",
random_state=3407
)

In [ ]:
import pandas as pd
import yaml
from datasets import Dataset

# ----------------------------
# 1. Load personality dataset
# ----------------------------
with open('/content/personality_dataset.json', 'r') as f:
    json_f = yaml.safe_load(f.read())

df = pd.DataFrame(json_f)
print("Columns loaded:", df.columns)

# ----------------------------
# 2. Alpaca prompt template
# ----------------------------
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Required to stop generation

# ----------------------------
# 3. Formatting function
# ----------------------------
def formatting_prompts_func(examples):
    texts = []

    for instruction, personality, input_text, output in zip(
        examples["instruction"],
        examples["personality"],
        examples["input"],
        examples["output"]
    ):

        # Merge personality + input into a single "Input"
        combined_input = f"Personality Type: {personality}\n\n{input_text}"

        # Format using Alpaca template
        formatted = alpaca_prompt.format(
            instruction,
            combined_input,
            output
        ) + EOS_TOKEN

        texts.append(formatted)

    return {"text": texts}

# ----------------------------
# 4. Convert to HuggingFace dataset
# ----------------------------
dataset = Dataset.from_pandas(df)

# ----------------------------
# 5. Map formatting onto dataset
# ----------------------------
dataset = dataset.map(formatting_prompts_func, batched=True)

# Test sample
print(dataset[0]["text"])


In [ ]:
print(dataset.column_names)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"

from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer
import torch
import os
from glob import glob
import matplotlib.pyplot as plt

training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    num_train_epochs = 5,
    learning_rate = 5e-5,
    warmup_ratio = 0.1,
    fp16 = False,
    bf16 = True,
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    save_strategy = "steps",
    save_steps = 20,
    save_total_limit = None,
    load_best_model_at_end = False,
    report_to = "none",
    seed = 3407
)
epoch_metrics = {
    "epoch": [],
    "train_loss": []
}

class EpochPlotCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.log_history:
            last_log = state.log_history[-1]
            if "loss" in last_log:
                epoch_metrics["epoch"].append(state.epoch)
                epoch_metrics["train_loss"].append(last_log["loss"])

        plt.figure()
        plt.plot(epoch_metrics["epoch"], epoch_metrics["train_loss"])
        plt.xlabel("Epoch")
        plt.ylabel("Training Loss")
        plt.title("LoRA Training Loss per Epoch")
        plt.grid(True)
        plt.savefig(f"{OUTPUT_DIR}/loss_epoch_{int(state.epoch)}.png")
        plt.close()
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = training_args
)

trainer.add_callback(EpochPlotCallback())

checkpoints = sorted(glob(f"{OUTPUT_DIR}/checkpoint-*"), key=os.path.getmtime)

if checkpoints:
    last_checkpoint = checkpoints[-1]
    print("🔄 Resuming training from:", last_checkpoint)
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh training...")
    trainer.train()

print("Training completed or safely paused.")
print("All checkpoints preserved in Google Drive.")
print(" Loss graphs saved per epoch in LoRA_outputs folder.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import os

# ================================
# 1. Collect training loss data
# ================================

# If Unsloth prints loss per step to a JSON or CSV, load it; otherwise, manually create list from log
# Example: if you saved logs as JSON like [{"step": 5, "loss": 1.232}, ...]
LOG_PATH = "/content/drive/MyDrive/LoRA_outputs_final/training_log.json"

if os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r") as f:
        log_data = json.load(f)
    df_loss = pd.DataFrame(log_data)
else:
    # Example: manually parse your log into pandas
    # Replace the lists below with your actual training steps & loss
    steps = list(range(5, 2115, 5))  # From your log: every 5 steps
    loss = [
        1.232100,1.225500,1.243400,1.220900,1.200900,1.196700,1.205500,1.190100,
        1.141700,1.105500,1.062100,1.008600,0.958800,0.908900,0.858100,0.797600
        # ... continue till last step (2110)
    ]
    df_loss = pd.DataFrame({"step": steps[:len(loss)], "loss": loss})

# ================================
# 2. Plot Loss Curve
# ================================

sns.set(style="whitegrid")
plt.figure(figsize=(12,6))
sns.lineplot(x="step", y="loss", data=df_loss, marker="o", color="red")
plt.title("LoRA Training Loss Curve")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.tight_layout()

# Save plot to Google Drive
SAVE_PATH = "/content/drive/MyDrive/LoRA_outputs_final/lora_training_loss.png"
plt.savefig(SAVE_PATH, dpi=300)
plt.show()

print(f"✅ Training loss curve saved at: {SAVE_PATH}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_loss['loss_smooth'] = df_loss['loss'].rolling(10).mean()

plt.figure(figsize=(12,6))
sns.lineplot(x="step", y="loss", data=df_loss, label="Raw Loss", color='red')
sns.lineplot(x="step", y="loss_smooth", data=df_loss, label="Smoothed Loss", color='blue')
plt.title("LoRA Training Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/LoRA_outputs_final/lora_training_loss_smooth.png", dpi=300)
plt.show()


In [ ]:
class ResearchStyleLossPlotCallback(TrainerCallback):
    def __init__(self, smoothing_window=50):
        self.steps = []
        self.losses = []
        self.smooth_losses = []
        self.epoch_losses = []
        self.epoch_steps = []
        self.smoothing_window = smoothing_window

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            step = state.global_step
            loss = logs["loss"]
            self.steps.append(step)
            self.losses.append(loss)

            # Moving average for smoothing
            if len(self.losses) >= self.smoothing_window:
                smooth = np.mean(self.losses[-self.smoothing_window:])
            else:
                smooth = np.mean(self.losses)
            self.smooth_losses.append(smooth)

    def on_epoch_end(self, args, state, control, **kwargs):
        # Record epoch-level loss
        if state.log_history:
            last_log = state.log_history[-1]
            if "loss" in last_log:
                self.epoch_losses.append(last_log["loss"])
                self.epoch_steps.append(state.global_step)

    def on_train_end(self, args, state, control, **kwargs):
        # Create a research-style figure
        plt.figure(figsize=(12,6))

        # 1. Raw loss
        plt.plot(self.steps, self.losses, color='lightgray', label='Raw Loss')

        # 2. Smoothed loss
        plt.plot(self.steps, self.smooth_losses, color='blue', linewidth=2, label=f'Smoothed Loss ({self.smoothing_window})')

        # 3. Epoch-level points
        plt.scatter(self.epoch_steps, self.epoch_losses, color='red', marker='o', label='Epoch Loss')

        # Labels, grid, legend
        plt.xlabel("Training Steps")
        plt.ylabel("Training Loss")
        plt.title("LoRA Training Progress (Research-style)")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend()
        plt.tight_layout()

        # Optional: log scale if needed
        # plt.yscale('log')

        plt.show()  # Inline in Colab
checkpoints = sorted(glob("/content/drive/MyDrive/LoRA_outputs_final/checkpoint-*"), key=os.path.getmtime)
if checkpoints:
    print("🔄 Resuming from:", checkpoints[-1])
    trainer.train(resume_from_checkpoint=checkpoints[-1])
else:
    print("🚀 Starting fresh training...")
    trainer.train()

print("✅ Training complete. Inline research-style graph displayed above.")

# BERT Evalution

In [ ]:
# ================================
# DATA SPLIT + GENERATION + EVALUATION
# ================================

from sklearn.model_selection import train_test_split
import json
import torch
from glob import glob
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# ================================
# 1. Split Dataset
# ================================

with open("/content/personality_dataset.json", "r") as f:
    full_data = json.load(f)

train_data, temp_data = train_test_split(full_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

with open("/content/train.json", "w") as f:
    json.dump(train_data, f, indent=2)

with open("/content/val.json", "w") as f:
    json.dump(val_data, f, indent=2)

with open("/content/test.json", "w") as f:
    json.dump(test_data, f, indent=2)

print("✅ Dataset split completed:")
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

# ================================
# 2. Load ONE sample from TEST set
# ================================

with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

sample = test_samples[3]  # 4th sample
reference = sample["output"]

test_prompt_alpaca = f"""
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
"""

# ================================
# 3. Load Latest LoRA Checkpoint
# ================================

CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
checkpoints = sorted(glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
latest_checkpoint = checkpoints[-1]

print("✅ Using latest checkpoint:", latest_checkpoint)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto"
)

model = FastLanguageModel.for_inference(model)
model.eval()

# ================================
# 4. Generate Output
# ================================

inputs = tokenizer([test_prompt_alpaca], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )

gen = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n===== MODEL OUTPUT =====\n")
print(gen)

# ================================
# 5. Load SBERT
# ================================

sbert_model = SentenceTransformer("all-mpnet-base-v2")
print("✅ SBERT Loaded")

# ================================
# 6. BERTScore Evaluation
# ================================

P, R, F1 = bert_score(
    [gen],
    [reference],
    model_type="microsoft/deberta-xlarge-mnli",
    lang="en"
)

print("\n===== FINAL EVALUATION =====")
print("BERT Precision:", P.item())
print("BERT Recall:", R.item())
print("BERT F1:", F1.item())

# ================================
# 7. SBERT Semantic Similarity
# ================================

gen_emb = sbert_model.encode(gen, convert_to_tensor=True)
ref_emb = sbert_model.encode(reference, convert_to_tensor=True)
sbert_sim = util.pytorch_cos_sim(gen_emb, ref_emb).item()

print("SBERT Similarity:", sbert_sim)



# INFP After Fine tuning Recommendation and Results


In [ ]:
# ================================
# DATA SPLIT + GENERATION + EVALUATION
# ================================

from sklearn.model_selection import train_test_split
import json
import torch
from glob import glob
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# ================================
# 1. Split Dataset
# ================================

with open("/content/personality_dataset.json", "r") as f:
    full_data = json.load(f)

train_data, temp_data = train_test_split(full_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

with open("/content/train.json", "w") as f:
    json.dump(train_data, f, indent=2)

with open("/content/val.json", "w") as f:
    json.dump(val_data, f, indent=2)

with open("/content/test.json", "w") as f:
    json.dump(test_data, f, indent=2)

print("✅ Dataset split completed:")
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))



with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

# Find the sample with personality 'INFP'
sample = next(s for s in test_samples if s["personality"] == "INFP")
reference = sample["output"]

test_prompt_alpaca = f"""
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
"""
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
checkpoints = sorted(glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
latest_checkpoint = checkpoints[-1]

print("✅ Using latest checkpoint:", latest_checkpoint)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto"
)

model = FastLanguageModel.for_inference(model)
model.eval()

inputs = tokenizer([test_prompt_alpaca], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
gen = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n===== MODEL OUTPUT =====\n")
print(gen)






INFP bert score (best references)

In [ ]:
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import torch
import pandas as pd
import matplotlib.pyplot as plt


# --- Load SBERT model ---
model_sbert = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")

# --- Example data ---
# For single reference:
generated = gen  # from your LoRA output
refs = [reference]  # can be a list of multiple references if available

# --- Compute BERTScore with best reference ---
# If multiple references exist, pick the closest via SBERT
best_ref = max(refs, key=lambda r: util.cos_sim(
    model_sbert.encode(r, convert_to_tensor=True),
    model_sbert.encode(generated, convert_to_tensor=True)
).item())

P, R, F1 = bert_score([generated], [best_ref], lang="en")
P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

# --- SBERT Semantic Alignment ---
emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
emb_gen = model_sbert.encode([generated], convert_to_tensor=True)
cos_scores = util.cos_sim(emb_gen, emb_refs)
sbert_alignment = cos_scores.max().item()

# --- Store results ---
results = [{
    "BERTScore_P": round(P, 3),
    "BERTScore_R": round(R, 3),
    "BERTScore_F1": round(F1, 3),

}]

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)


Top K reference

In [ ]:
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import torch
import pandas as pd

# --- Load SBERT model ---
model_sbert = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")

# --- Example data ---
generated = gen  # your LoRA output
refs = all_refs  # list of all reference outputs

# --- Encode all references and generated text ---
emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
emb_gen = model_sbert.encode([generated], convert_to_tensor=True)

# --- Compute cosine similarity ---
cos_scores = util.cos_sim(emb_gen, emb_refs)[0]

# --- Pick top 3 references ---
top3_idx = torch.topk(cos_scores, k=3).indices.tolist()
top3_refs = [refs[i] for i in top3_idx]

# --- Compute BERTScore for top 3 refs ---
bert_scores = []
for ref in top3_refs:
    P, R, F1 = bert_score([generated], [ref], lang="en")
    bert_scores.append({
        "Reference": ref,
        "SBERT_Sim": round(util.cos_sim(emb_gen, model_sbert.encode([ref], convert_to_tensor=True)).item(), 3),
        "BERT_P": round(float(P[0]), 3),
        "BERT_R": round(float(R[0]), 3),
        "BERT_F1": round(float(F1[0]), 3)
    })

# --- Display results ---
df_metrics = pd.DataFrame(bert_scores)
display(df_metrics)


Semantic Rough and BLEU (INFP Personality type)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from nltk.util import ngrams
import numpy as np
import re
import pandas as pd

sbert_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic n-gram P-BLEU ---
def semantic_p_bleu(hypo_text, refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = sbert_model.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = sbert_model.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L ---
def semantic_rouge_l(hypo_text, refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = sbert_model.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = sbert_model.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluate using your variables ---
reference_list = [reference]  # can have multiple references
gen_list = [gen]  # can evaluate multiple generated outputs

results = []
for g in gen_list:
    p_bleu = semantic_p_bleu(g, reference_list)
    rouge_l = semantic_rouge_l(g, reference_list)
    sbert_sim = util.cos_sim(
        sbert_model.encode([g], convert_to_tensor=True),
        sbert_model.encode(reference_list, convert_to_tensor=True)
    )[0,0].item()

    results.append({
        "Semantic P-BLEU": round(p_bleu, 3),
        "Semantic ROUGE-L": round(rouge_l, 3),
        "SBERT Similarity": round(sbert_sim, 3)
    })

# Display results in a table
df = pd.DataFrame(results)
display(df)


# Explainable Recommendation

In [ ]:


import re
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# ================================
# 1. Full Input Sample
# ================================
sample_input = {
    "personality": "INFP",
    "instruction": "Generate a personality-aligned home interior design recommendation, integrating the given design features and personality traits. Ensure the design reflects psychological alignment, color harmony, and coherent style preferences.",
    "input": """
- Style: modern
  Palette: grey, darkslategrey, lightgray, darkolivegreen, tan
  Description: a living room with large window and couch
- Style: modern
  Palette: silver, darkolivegreen, sienna, rosybrown, saddlebrown
  Description: a living room with couch television and flat screen
- Style: modern
  Palette: grey, darkslategrey, oldlace, palegoldenrod, darkslategrey
  Description: a living room with couches and television
- Style: modern
  Palette: darkgray, black, gainsboro, darkolivegreen, darkolivegreen
  Description: a living room with large glass door
- Style: modern
  Palette: silver, darkolivegreen, sienna, dimgray, dimgray
  Description: a living room with couch and television
- Style: modern
  Palette: darkslategrey, gainsboro, sienna, peru, lightslategrey
  Description: a living room with couches and fireplace
- Style: modern
  Palette: black, silver, dimgray, lightslategrey, lightslategrey
  Description: a living room with couch chair and table
- Style: modern
  Palette: darkgray, darkslategrey, darkolivegreen, gainsboro, darkolivegreen
  Description: a restaurant with tables and chairs
- Style: modern
  Palette: grey, black, darkolivegreen, silver, tan
  Description: a living room with couches television and flat screen
- Style: modern
  Palette: darkslategrey, lightgray, grey, darkgray, darkgray
  Description: a living room with couch chair and coffee table
- Style: modern
  Palette: silver, darkslategrey, dimgray, sienna, darkolivegreen
  Description: a kitchen with wooden counter top and ceiling
- Style: modern
  Palette: darkslategrey, lightgray, grey, dimgray, grey
  Description: a living room with couch television and
- Style: modern
  Palette: darkgray, black, darkolivegreen, darkolivegreen, dimgray
  Description: a living room with couch television and table
- Style: modern
  Palette: darkslategrey, lightgray, grey, darkkhaki, lightslategrey
  Description: a living room with couch chair and coffee table
- Style: modern
  Palette: darkslategrey, silver, darkgray, grey, grey
  Description: a living room with couch chair and television
- Style: modern
  Palette: silver, darkslategrey, sienna, dimgray, grey
  Description: a living room with couch and television
- Style: modern
  Palette: darkgray, darkslategrey, dimgray, darkslategrey, darkolivegreen
  Description: a living room with fireplace and couches
- Style: modern
  Palette: wheat, saddlebrown, peru, darksalmon, dimgray
  Description: a living room with couch television and fireplace
- Style: vintage
  Palette: sienna, black, silver, darksalmon, burlywood
  Description: a living room with fireplace and couches
- Style: modern
  Palette: grey, gainsboro, black, silver, darkgray
  Description: a bedroom with bed chair and mirror
"""
}

# ================================
# 2. Full Generated Recommendation
# ================================
generated_recommendation = """
Based on the provided design inputs, I've identified some common themes and patterns to create a personality-aligned home interior design recommendation.

Personality Traits:
The individual appears to value modernity, simplicity, and functionality. They seem to appreciate a calm and serene atmosphere, often incorporating natural elements and earthy tones into their designs. Comfort and coziness are also prioritized.

Color Palette:
Dominant colors include various shades of grey, dark slate grey, and dark olive green. Neutral tones like silver, dim grey, and gainsboro complement these. Accent colors such as sienna, dark khaki, and dark salmon add warmth and depth.

Design Features:
The design often features clean lines, minimal ornamentation, and functional furniture. Emphasis is placed on creating a sense of flow and cohesion between different areas of the home. Natural materials like wood and stone add texture and warmth.

Recommended Design:

Living Room:
- Walls: Dark slate grey (primary color)
- Furniture: A sleek, low-profile sofa in a neutral tone like grey or beige
- Accent wall: A statement wall featuring dark olive green artwork or a geometric pattern
- Flooring: Light grey or beige hardwood floors
- Decor: Add warmth with natural textiles like woven baskets, jute rugs, and a statement piece of artwork or sculpture

Kitchen:
- Cabinets: Dark slate grey or black cabinets with white or light grey countertops
- Appliances: Stainless steel appliances or silver accents to add a touch of modernity
- Lighting: Under-cabinet lighting to create a warm and inviting ambiance

Bedroom:
- Walls: A soothing grey or beige tone to create a calming atmosphere
- Furniture: A comfortable, ergonomic bed with a minimalist headboard
- Decor: Add warmth with natural textiles like linen curtains, a jute rug, and a few carefully selected pieces of artwork or sculptures

Throughout the home, incorporate natural elements like potted plants, a statement piece of artwork or sculpture, and a mix of textures to create a cozy and inviting atmosphere.
"""

# ================================
# 3. Load SBERT Model
# ================================
device = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer("all-mpnet-base-v2", device=device)

# ================================
# 4. Extract Phrases (n-grams)
# ================================
def extract_ngrams(text, max_n=2):
    tokens = re.findall(r'\b\w+\b', text.lower())
    ngrams = tokens[:]  # unigrams
    for n in range(2, max_n + 1):
        ngrams += [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return ngrams

input_text = sample_input['instruction'] + " " + sample_input['input']
phrases = extract_ngrams(input_text, max_n=2)

# ================================
# 5. Compute Semantic Contribution Scores
# ================================
phrase_embs = sbert.encode(phrases, convert_to_tensor=True)
gen_emb = sbert.encode([generated_recommendation], convert_to_tensor=True)

cosine_scores = util.cos_sim(phrase_embs, gen_emb).squeeze()

# Normalize to 0-1
weights = (cosine_scores - cosine_scores.min()) / (cosine_scores.max() - cosine_scores.min())
weighted_phrases = list(zip(phrases, weights.tolist()))

# Remove duplicates, keep highest score
phrase_dict = {}
for phrase, score in weighted_phrases:
    if phrase not in phrase_dict or score > phrase_dict[phrase]:
        phrase_dict[phrase] = score

# Sort descending by score
weighted_phrases_unique = sorted(phrase_dict.items(), key=lambda x: x[1], reverse=True)

# Top 15 contributing phrases
top_k = 30
top_contributing = weighted_phrases_unique[:top_k]

# ================================
# 6. Display as Table
# ================================
df = pd.DataFrame(top_contributing, columns=["Phrase", "Influence Score"])
df["Influence Score"] = df["Influence Score"].round(3)
df.to_csv("top_contributing_phrases.csv", index=False)
print(" Top contributing phrases saved to top_contributing_phrases.csv")
# ================================
# 7. Homeowner-Friendly Explanation
# ================================
explanation = f"""
Dear Homeowner,

Based on the information you provided, we generated the following interior design recommendation for you:

{generated_recommendation}

This recommendation was primarily influenced by the following key aspects in your input, ranked by their contribution:
"""

for phrase, score in top_contributing:
    explanation += f"- {phrase} (importance: {score:.2f})\n"

explanation += """
These factors guided our choice of style, color palette, furniture, and layout to ensure your home reflects a modern, calm, cozy, and personality-aligned aesthetic.
"""

print("\n===== EXPLAINABLE RECOMMENDATION TO HOMEOWNER =====\n")
print(explanation)


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 8))
plt.barh([p[0] for p in reversed(top_contributing)], [p[1] for p in reversed(top_contributing)], color='teal')
plt.xlabel("Influence Score")
plt.title("Top 30 Contributing Words/Phrases to Recommendation")
plt.gca().invert_yaxis()  # highest score at top
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ================================
# 1. Load Top Contributing Words for Recommendation
# ================================
top_contrib_file = "top_contributing_phrases.csv"  # Phrase, Influence Score
top_contrib_df = pd.read_csv(top_contrib_file)
top_contrib_df = top_contrib_df.round({'Influence Score': 3})

# Keep top 30 unique phrases
top_contrib_df = top_contrib_df.drop_duplicates(subset=['Phrase']).head(30)
top_contrib_str = "<br>".join([f"{row.Phrase} (score: {row['Influence Score']})" for _, row in top_contrib_df.iterrows()])

# ================================
# 2. Load Words Influencing Personality Prediction
# ================================
personality_file = "INFP_word_shap.csv"  # Predicted_Personality, Word, Impact, Direction
personality_df = pd.read_csv(personality_file)
personality_df = personality_df.round({'Impact': 3})

# Keep only INFP words
personality_df = personality_df[personality_df['Predicted_Personality'] == 'INFP']

# Convert Word column to string (to avoid errors with str.contains)
personality_df['Word'] = personality_df['Word'].astype(str)

# Remove words with negative Impact or containing '.', ',', or '-'
personality_df = personality_df[(personality_df['Impact'] >= 0) &
                                (~personality_df['Word'].str.contains(r"[.,-]"))]

# Keep top 30 unique words
personality_df = personality_df.drop_duplicates(subset=['Word']).head(30)

# Convert to display string
personality_shap_str = "<br>".join([
    f"{row.Word} (impact: {row.Impact}, dir: {row.Direction})"
    for _, row in personality_df.iterrows()
])


# ================================
# 3. Predicted Personality Type
# ================================
predicted_personality = "INFP"

# ================================
# 4. Generated Recommendation
# ================================
generated_recommendation = """
Based on the provided design inputs, I've identified some common themes and patterns to create a personality-aligned home interior design recommendation.

Personality Traits:
The individual appears to value modernity, simplicity, and functionality. They seem to appreciate a calm and serene atmosphere, often incorporating natural elements and earthy tones into their designs. Comfort and coziness are also prioritized.

Color Palette:
Dominant colors include various shades of grey, dark slate grey, and dark olive green. Neutral tones like silver, dim grey, and gainsboro complement these. Accent colors such as sienna, dark khaki, and dark salmon add warmth and depth.

Design Features:
The design often features clean lines, minimal ornamentation, and functional furniture. Emphasis is placed on creating a sense of flow and cohesion between different areas of the home. Natural materials like wood and stone add texture and warmth.

Recommended Design:

Living Room:
- Walls: Dark slate grey (primary color)
- Furniture: A sleek, low-profile sofa in a neutral tone like grey or beige
- Accent wall: A statement wall featuring dark olive green artwork or a geometric pattern
- Flooring: Light grey or beige hardwood floors
- Decor: Add warmth with natural textiles like woven baskets, jute rugs, and a statement piece of artwork or sculpture

Kitchen:
- Cabinets: Dark slate grey or black cabinets with white or light grey countertops
- Appliances: Stainless steel appliances or silver accents to add a touch of modernity
- Lighting: Under-cabinet lighting to create a warm and inviting ambiance

Bedroom:
- Walls: A soothing grey or beige tone to create a calming atmosphere
- Furniture: A comfortable, ergonomic bed with a minimalist headboard
- Decor: Add warmth with natural textiles like linen curtains, a jute rug, and a few carefully selected pieces of artwork or sculptures

Throughout the home, incorporate natural elements like potted plants, a statement piece of artwork or sculpture, and a mix of textures to create a cozy and inviting atmosphere.
"""

# ================================
# 5. Create Beautiful Plotly Table
# ================================
fig = go.Figure(data=[go.Table(
    columnwidth=[120, 800],
    header=dict(values=["Category", "Details"],
                fill_color='darkslategray',
                font=dict(color='white', size=16),
                align='left'),
    cells=dict(values=[
        ["Predicted Personality Type",
         "Generated Recommendation",
         "Top Words Contributing to Recommendation",
         "Words Influencing Personality Prediction"],
        [predicted_personality,
         generated_recommendation,
         top_contrib_str,
         personality_shap_str]],
        fill_color='lavender',
        align='left',
        font=dict(color='black', size=14),
        height=45
    ))
])

fig.update_layout(
    title="Explainable Home Interior Recommendation Summary",
    title_font_size=20,
    width=1300,
    height=1000
)

fig.show()

# ================================
# 6. Save as HTML
# ================================
fig.write_html("explainable_recommendation_summary.html")
print("✅ Saved interactive summary table as explainable_recommendation_summary.html")


# Test Case 1: ISTP personality (Recommendation generation and evalution)

In [ ]:


from sklearn.model_selection import train_test_split
import json
import torch
from glob import glob
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# ================================
# 1. Split Dataset
# ================================

with open("/content/personality_dataset.json", "r") as f:
    full_data = json.load(f)

train_data, temp_data = train_test_split(full_data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

with open("/content/train.json", "w") as f:
    json.dump(train_data, f, indent=2)

with open("/content/val.json", "w") as f:
    json.dump(val_data, f, indent=2)

with open("/content/test.json", "w") as f:
    json.dump(test_data, f, indent=2)

print("✅ Dataset split completed:")
print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

# ================================
# 2. Load ONE sample from TEST set
# ================================

with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

# Find the sample with personality 'INFP'
sample = next(s for s in test_samples if s["personality"] == "ISTP")
reference_istp = sample["output"]

test_prompt_alpaca = f"""
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
"""
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
checkpoints = sorted(glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
latest_checkpoint = checkpoints[-1]

print("✅ Using latest checkpoint:", latest_checkpoint)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto"
)

model = FastLanguageModel.for_inference(model)
model.eval()

inputs = tokenizer([test_prompt_alpaca], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
gen_istp = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n===== MODEL OUTPUT =====\n")
print(gen_istp)






BERT score evalution most optimal ref

In [ ]:
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import torch
import pandas as pd
import matplotlib.pyplot as plt

# --- Load SBERT model ---
sbert_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")
reference_istp
# --- Example data ---
# For single reference:
generated = gen_istp  # from your LoRA output
refs = [reference_istp]  # can be a list of multiple references if available

# --- Compute BERTScore with best reference ---
# If multiple references exist, pick the closest via SBERT
best_ref = max(refs, key=lambda r: util.cos_sim(
    model_sbert.encode(r, convert_to_tensor=True),
    model_sbert.encode(generated, convert_to_tensor=True)
).item())

P, R, F1 = bert_score([generated], [best_ref], lang="en")
P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

# --- SBERT Semantic Alignment ---
emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
emb_gen = model_sbert.encode([generated], convert_to_tensor=True)
cos_scores = util.cos_sim(emb_gen, emb_refs)
sbert_alignment = cos_scores.max().item()

# --- Store results ---
results = [{
    "BERTScore_P": round(P, 3),
    "BERTScore_R": round(R, 3),
    "BERTScore_F1": round(F1, 3),

}]

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)


Top k references

In [ ]:
import json
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# ---------------- LOAD TEST DATA ----------------
with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

# Get ISTP sample
sample_istp = next(s for s in test_samples if s["personality"] == "ISTP")

# ✅ All references for ISTP
all_refs_istp = sample_istp.get("references", [sample_istp["output"]])

# ✅ YOUR LoRA GENERATED TEXT
generated = gen_istp

refs = all_refs_istp

# ---------------- LOAD SBERT (CPU safe) ----------------
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

# ---------------- ENCODE ----------------
emb_refs = sbert_model.encode(refs, convert_to_tensor=True)
emb_gen = sbert_model.encode([generated], convert_to_tensor=True)

# ---------------- COSINE SIMILARITY ----------------
cos_scores = util.cos_sim(emb_gen, emb_refs)[0]

# ✅ Select BEST 3 references
k = min(3, len(refs))
top3_idx = torch.topk(cos_scores, k=k).indices.tolist()
top3_refs = [refs[i] for i in top3_idx]
top3_sims = [cos_scores[i].item() for i in top3_idx]

# ---------------- BERTSCORE (CPU ONLY = no crash) ----------------
bert_results = []

for ref, sim in zip(top3_refs, top3_sims):
    P, R, F1 = bert_score(
        [generated],
        [ref],
        lang="en",
        device="cpu",                       # ✅ prevents CUDA OOM
        model_type="distilbert-base-uncased"
    )

    bert_results.append({
        "SBERT_Similarity": round(sim, 3),
        "BERT_P": round(float(P[0]), 3),
        "BERT_R": round(float(R[0]), 3),
        "BERT_F1": round(float(F1[0]), 3),
        "Reference Preview": ref[:150] + "..."
    })

# ---------------- DISPLAY ----------------
df_metrics = pd.DataFrame(bert_results)
display(df_metrics)


In [ ]:
import json
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score

# ---------------- LOAD TEST DATA ----------------
with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

sample_istp = next(s for s in test_samples if s["personality"] == "ISTP")
all_refs_istp = sample_istp.get("references", [sample_istp["output"]])

generated = gen_istp
refs = all_refs_istp

# ---------------- SBERT ----------------
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

emb_refs = sbert_model.encode(refs, convert_to_tensor=True)
emb_gen = sbert_model.encode([generated], convert_to_tensor=True)

cos_scores = util.cos_sim(emb_gen, emb_refs)[0]

k = min(3, len(refs))
top3_idx = torch.topk(cos_scores, k=k).indices.tolist()
top3_refs = [refs[i] for i in top3_idx]
top3_sims = [cos_scores[i].item() for i in top3_idx]

# ---------------- BERTSCORE ----------------
bert_results = []

print("\n================ TOP 3 MATCHED REFERENCES ================\n")

for i, (ref, sim) in enumerate(zip(top3_refs, top3_sims), start=1):
    P, R, F1 = bert_score(
        [generated],
        [ref],
        lang="en",
        device="cpu",
        model_type="distilbert-base-uncased"
    )

    print(f"--- TOP {i} REFERENCE ---")
    print("Reference Text:\n", ref)
    print("SBERT Similarity:", round(sim, 4))
    print("BERT Precision:", round(float(P[0]), 4))
    print("BERT Recall   :", round(float(R[0]), 4))
    print("BERT F1 Score :", round(float(F1[0]), 4))
    print("--------------------------------------------------------\n")

    bert_results.append({
        "Rank": i,
        "SBERT_Similarity": round(sim, 3),
        "BERT_P": round(float(P[0]), 3),
        "BERT_R": round(float(R[0]), 3),
        "BERT_F1": round(float(F1[0]), 3),
        "Reference": ref
    })

# ---------------- TABLE OUTPUT ----------------
df_metrics = pd.DataFrame(bert_results)
display(df_metrics)


Semantic P-Bleu and Rouge-l

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from nltk.util import ngrams
import numpy as np
import re
import pandas as pd
import json

# --- Load test dataset ---
with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

# --- Select a sample with ISTP personality ---
sample_istp = next(s for s in test_samples if s["personality"] == "ISTP")


# --- Define all references ---
# If the dataset has multiple references, make a list
all_refs_istp = sample_istp.get("references", [sample_istp["output"]])

# --- Load SBERT model ---
# Use CPU if GPU memory is limited
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic n-gram P-BLEU ---
def semantic_p_bleu(hypo_text, refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = sbert_model.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = sbert_model.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L ---
def semantic_rouge_l(hypo_text, refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = sbert_model.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = sbert_model.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluate using your variables ---
reference_list = all_refs_istp  # multiple references for ISTP personality
gen_list = [gen_istp]           # your generated LoRA output

results = []
for g in gen_list:
    p_bleu = semantic_p_bleu(g, reference_list)
    rouge_l = semantic_rouge_l(g, reference_list)
    sbert_sim = util.cos_sim(
        sbert_model.encode([g], convert_to_tensor=True),
        sbert_model.encode(reference_list, convert_to_tensor=True)
    ).max().item()  # best similarity among multiple references

    results.append({
        "Semantic P-BLEU": round(p_bleu, 3),
        "Semantic ROUGE-L": round(rouge_l, 3),
        "SBERT Similarity": round(sbert_sim, 3)
    })

# Display results in a table
df = pd.DataFrame(results)
display(df)


# Test Case 2:ENFP

In [ ]:
from sklearn.model_selection import train_test_split
import json
import torch
from glob import glob
from unsloth import FastLanguageModel

import gc, os
torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ================= LOAD DATA =================
with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

sample = next(s for s in test_samples if s["personality"] == "ENFP")
reference_enfp = sample["output"]

test_prompt_alpaca = f"""
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
"""


# ================= LOAD LATEST CHECKPOINT =================
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
checkpoints = sorted(glob(f"{CHECKPOINT_DIR}/checkpoint-*"), key=lambda x: int(x.split("-")[-1]))
latest_checkpoint = checkpoints[-1]

print("✅ Using latest checkpoint:", latest_checkpoint)


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = latest_checkpoint,
    max_seq_length = 512,          # ✅ reduced from 1024
    dtype = torch.bfloat16,        # ✅ stable & efficient
    load_in_4bit = True,           # ✅ required for memory safety
    device_map = "cuda"
)

model = FastLanguageModel.for_inference(model)
model.eval()


# ================= GENERATION =================
inputs = tokenizer(
    test_prompt_alpaca,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to("cuda")


with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )

gen_enfp = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n===== MODEL OUTPUT =====\n")
print(gen_enfp)


In [ ]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )

gen_enfp = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n===== MODEL OUTPUT =====\n")
print(gen_enfp)

Semantic Bleu and rough for ENFP

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from nltk.util import ngrams
import numpy as np
import re
import pandas as pd
import json

# --- Load test dataset ---
with open("/content/test.json", "r") as f:
    test_samples = json.load(f)

# --- Select a sample with ISTP personality ---
sample_enfp = next(s for s in test_samples if s["personality"] == "ENFP")


# --- Define all references ---
# If the dataset has multiple references, make a list
all_refs_enfp = sample_enfp.get("references", [sample_istp["output"]])

# --- Load SBERT model ---
# Use CPU if GPU memory is limited
sbert_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic n-gram P-BLEU ---
def semantic_p_bleu(hypo_text, refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = sbert_model.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = sbert_model.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L ---
def semantic_rouge_l(hypo_text, refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = sbert_model.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = sbert_model.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluate using your variables ---
reference_list = all_refs_enfp
gen_list = [gen_enfp]           # your generated LoRA output

results = []
for g in gen_list:
    p_bleu = semantic_p_bleu(g, reference_list)
    rouge_l = semantic_rouge_l(g, reference_list)
    sbert_sim = util.cos_sim(
        sbert_model.encode([g], convert_to_tensor=True),
        sbert_model.encode(reference_list, convert_to_tensor=True)
    ).max().item()  # best similarity among multiple references

    results.append({
        "Semantic P-BLEU": round(p_bleu, 3),
        "Semantic ROUGE-L": round(rouge_l, 3),
        "SBERT Similarity": round(sbert_sim, 3)
    })

# Display results in a table
df = pd.DataFrame(results)
display(df)


In [ ]:
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import torch
import pandas as pd
import matplotlib.pyplot as plt

# --- Load SBERT model ---
sbert_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")
reference_istp
# --- Example data ---
# For single reference:
reference_list = all_refs_enfp
gen_list = [gen_enfp]


# --- Compute BERTScore with best reference ---
# If multiple references exist, pick the closest via SBERT
best_ref = max(refs, key=lambda r: util.cos_sim(
    model_sbert.encode(r, convert_to_tensor=True),
    model_sbert.encode(generated, convert_to_tensor=True)
).item())

P, R, F1 = bert_score([generated], [best_ref], lang="en")
P, R, F1 = float(P[0]), float(R[0]), float(F1[0])

# --- SBERT Semantic Alignment ---
emb_refs = model_sbert.encode(refs, convert_to_tensor=True)
emb_gen = model_sbert.encode([generated], convert_to_tensor=True)
cos_scores = util.cos_sim(emb_gen, emb_refs)
sbert_alignment = cos_scores.max().item()

# --- Store results ---
results = [{
    "BERTScore_P": round(P, 3),
    "BERTScore_R": round(R, 3),
    "BERTScore_F1": round(F1, 3),

}]

# --- Create DataFrame ---
df_metrics = pd.DataFrame(results)
display(df_metrics)


# SBERT Similarity ,P-Bleu and P-Rough INFP

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from nltk.util import ngrams
import numpy as np
import re
import pandas as pd

sbert_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cuda")

# --- Text normalization ---
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- Semantic n-gram P-BLEU ---
def semantic_p_bleu(hypo_text, refs, n=3, sim_threshold=0.5):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    scores = []
    for ngram_size in range(1, n+1):
        hypo_ngrams = list(ngrams(hypo_tokens, ngram_size))
        hypo_str_ngrams = [" ".join(ng) for ng in hypo_ngrams]
        hypo_emb = sbert_model.encode(hypo_str_ngrams, convert_to_tensor=True)

        max_match_scores = []
        for ref_text in refs:
            ref_tokens = normalize_text(ref_text).split()
            ref_ngrams = list(ngrams(ref_tokens, ngram_size))
            ref_str_ngrams = [" ".join(ng) for ng in ref_ngrams]
            if not ref_str_ngrams:
                continue
            ref_emb = sbert_model.encode(ref_str_ngrams, convert_to_tensor=True)

            cos_sim = util.cos_sim(hypo_emb, ref_emb)
            max_per_hypo = cos_sim.max(dim=1).values
            max_per_hypo = (max_per_hypo >= sim_threshold).float()
            max_match_scores.append(max_per_hypo.sum().item() / len(hypo_ngrams))
        scores.append(max(max_match_scores) if max_match_scores else 0)
    return np.mean(scores)

# --- Semantic ROUGE-L ---
def semantic_rouge_l(hypo_text, refs, sim_threshold=0.6):
    hypo_tokens = normalize_text(hypo_text).split()
    if not hypo_tokens:
        return 0.0
    max_score = 0.0
    hypo_emb = sbert_model.encode(hypo_tokens, convert_to_tensor=True)

    for ref_text in refs:
        ref_tokens = normalize_text(ref_text).split()
        if not ref_tokens:
            continue
        ref_emb = sbert_model.encode(ref_tokens, convert_to_tensor=True)
        cos_sim = util.cos_sim(hypo_emb, ref_emb)
        binary_matches = (cos_sim >= sim_threshold).float()
        precision = binary_matches.max(dim=1).values.mean().item()
        recall = binary_matches.max(dim=0).values.mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        max_score = max(max_score, f1)
    return max_score

# --- Evaluate using your variables ---
reference_list = [reference]  # can have multiple references
gen_list = [gen]  # can evaluate multiple generated outputs

results = []
for g in gen_list:
    p_bleu = semantic_p_bleu(g, reference_list)
    rouge_l = semantic_rouge_l(g, reference_list)
    sbert_sim = util.cos_sim(
        sbert_model.encode([g], convert_to_tensor=True),
        sbert_model.encode(reference_list, convert_to_tensor=True)
    )[0,0].item()

    results.append({
        "Semantic P-BLEU": round(p_bleu, 3),
        "Semantic ROUGE-L": round(rouge_l, 3),
        "SBERT Similarity": round(sbert_sim, 3)
    })

# Display results in a table
df = pd.DataFrame(results)
display(df)


# Generate recommendation using RAG prompt for ISTP Type (Testing Work)

In [ ]:
# ============================================
# 1️⃣ Mount Google Drive
# ============================================
print("Step 1️⃣: Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
DATASET_PATH = "/content/personality_data.json"  # adjust path if needed

# ============================================
# 2️⃣ Standard Imports
# ============================================
import pandas as pd
from bert_score import score as bert_score
from sentence_transformers import SentenceTransformer, util
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import nltk
import torch
from glob import glob
nltk.download("punkt")


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"S: Using device: {device}")

print("Step : Searching for latest checkpoint...")

# ============================================
#  Load Latest Checkpoint (Fixed for free Colab)
# ============================================

from unsloth import FastLanguageModel

checkpoint_path = "/content/drive/MyDrive/LoRA_outputs_final/checkpoint-1980"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = checkpoint_path,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
    device_map = "auto"   # let Unsloth decide placement safely
)

model = FastLanguageModel.for_inference(model)
model.eval()
print("Model loaded successfully without CPU/GPU dispatch errors!")



print(" Loading SBERT model for evaluation...")
sbert_model = SentenceTransformer("all-mpnet-base-v2")
print("SBERT loaded!")

test_prompt_alpaca = f"""
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Design a space that represents the user’s personality and nurtures their emotional needs.

### Input:
You are a visionary interior designer specializing in crafting dynamic, expressive spaces for high-energy personalities.

Your main design reference is:
Personality: ISTP| Style: modern | Palette: sienna, black, lightgray, darkgray, darkgray | Description: a living room with couches and table

You have also consulted the following related design inspirations:
1. Personality: ISTP | Style: modern | Palette: dimgray, lightgray, tan, black, darkgray | Description: a living room with couches and table
2. Personality: ISTP  | Style: modern | Palette: darkslategrey, lightgray, silver, grey, darkgray | Description: a living room with couches and table
3. Personality: ISTP  | Style: modern | Palette: dimgray, lightgray, black, silver, darkgray | Description: a living room with couches and coffee table

Analyze these references carefully. Then, creatively blend their dominant color trends, furniture ideas, and emotional vibes into a new, cohesive design recommendation.

In your response:
- Start by summarizing the common color themes and emotional tones found across all references.
- Then propose a bold new dominant color palette and explain why it captures adventurous energy.
- Suggest 2–3 signature furniture pieces that match the energetic and free-spirited personality.
- Evoke vivid emotions (like excitement, freedom, boldness) as if painting a living picture of the space.

 Think of yourself not just as a designer, but as an artist capturing the soul of a free-spirited adventurer.

### Response:
"""

# test_prompt_alpaca = f"""
# Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# Design a space that represents the user’s personality and nurtures their emotional needs.

# ### Input:
# Personality: INFP
# Primary Design Inspiration:
#  Style: modern | Palette: darkgray, darkslategrey, saddlebrown, sienna, dimgray | Description: a living room with couch and television

# Related Inspirations:
# 1. Style: modern | Palette: darkgray, darkslategrey, saddlebrown, sienna, dimgray | Description: a living room with television and couch
# 2. Style: modern | Palette: darkgray, black, sienna, saddlebrown, dimgray | Description: a living room with couch and television
# 3. Style: modern | Palette: darkgray, darkslategrey, sienna, saddlebrown, dimgray | Description: a living room with couch, chair, and television

# ### Response:
# """

inputs = tokenizer([test_prompt_alpaca], return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.85,
        top_p=0.92,
        do_sample=True,
        repetition_penalty=1.15
    )

gen = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(gen)


Evalute semantic BERT and SBERT

In [ ]:
# ============================================
# BERTScore + SBERT Evaluation (Individual High Semantic References)
# ============================================

import torch
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
import pandas as pd
import json
import re
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ---------- Load dataset ----------
DATA_PATH = "/content/personality_dataset.json"
with open(DATA_PATH, 'r') as f:
    data_json = json.load(f)
df = pd.DataFrame(data_json)

# ---------- Load SBERT ----------
sbert_model = SentenceTransformer("all-mpnet-base-v2", device=device)
print("SBERT loaded!")

# ---------- Text normalization ----------
def normalize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ---------- Encode batch ----------
def encode_batch(texts, model, batch_size=16):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, convert_to_tensor=True)
        embeddings.append(emb)
        torch.cuda.empty_cache()
        gc.collect()
    return torch.cat(embeddings)

# ---------- Evaluation function ----------
def evaluate_bert_sbert_each_highsim(gen_text, personality, threshold=0.6):
    gen_clean = normalize(gen_text)

    # Get all references for the personality
    ref_texts = df[df["personality"]==personality]["output"].tolist()
    if not ref_texts:
        return {"error": "No references found"}
    ref_clean = [normalize(r) for r in ref_texts]

    # SBERT embeddings
    gen_emb = sbert_model.encode([gen_clean], convert_to_tensor=True)
    ref_embs = encode_batch(ref_clean, sbert_model)

    # Cosine similarity
    sim_scores = util.cos_sim(gen_emb, ref_embs)[0].cpu().numpy()

    # Display all SBERT similarities
    print("\n===== All SBERT Similarities =====")
    for i, (ref, sim) in enumerate(zip(ref_clean, sim_scores)):
        print(f"{i}: {sim:.4f} | {ref[:80]}...")

    # Choose references above threshold
    high_sim_indices = [i for i, score in enumerate(sim_scores) if score >= threshold]
    if not high_sim_indices:
        return {"error": f"No references with similarity >= {threshold}"}

    high_refs = [ref_clean[i] for i in high_sim_indices]
    high_sims = [sim_scores[i] for i in high_sim_indices]

    # Display high similarity references
    print(f"\n===== References with SBERT >= {threshold} =====")
    for i, (ref, sim) in enumerate(zip(high_refs, high_sims)):
        print(f"{i}: {sim:.4f} | {ref[:80]}...")

    # Compute BERTScore for each high similarity reference individually
    bert_results = []
    for i, ref in enumerate(high_refs):
        P, R, F1 = bert_score([gen_clean], [ref],
                              model_type="microsoft/deberta-v3-base", lang="en")
        bert_results.append({
            "Ref_Index": high_sim_indices[i],
            "SBERT_Sim": high_sims[i],
            "BERT_P": P.item(),
            "BERT_R": R.item(),
            "BERT_F1": F1.item(),
            "Reference": ref
        })

    return bert_results

# ---------- Example usage ----------
results = evaluate_bert_sbert_each_highsim(gen, personality="ISTP", threshold=0.6)

print("\n===== BERT & SBERT Scores for Each High-Sim Reference =====")
pd.set_option('display.max_colwidth', 200)
df_results = pd.DataFrame(results)
display(df_results)


RAG Based prompt for INFP

In [ ]:


from glob import glob
import os
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from unsloth import FastLanguageModel
import json
import re
from torch.utils.data import DataLoader
import gc

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- Load dataset ----------
DATA_PATH = "/content/personality_dataset.json"
with open(DATA_PATH, 'r') as f:
    data_json = json.load(f)
df = pd.DataFrame(data_json)

# ---------- Load SBERT ----------
sbert_model = SentenceTransformer("all-mpnet-base-v2", device=device)
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# ---------- Text normalization ----------
def normalize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ---------- Encode batch ----------
def encode_batch(texts, model, batch_size=2):
    embeddings = []
    loader = DataLoader(texts, batch_size=batch_size, shuffle=False)
    for batch in loader:
        emb = model.encode(batch, convert_to_tensor=True)
        embeddings.append(emb)
        torch.cuda.empty_cache()
        gc.collect()
    return torch.cat(embeddings)

# ---------- Weighted BERTScore ----------
def bert_score_weighted(gen_clean, ref_clean, weights, batch_size=2):
    P_list, R_list, F1_list = [], [], []
    for i in range(0, len(ref_clean), batch_size):
        batch_refs = ref_clean[i:i+batch_size]
        P, R, F1 = bert_score([gen_clean]*len(batch_refs), batch_refs, model_type="microsoft/deberta-v3-base", lang="en")
        P_list.extend(P)
        R_list.extend(R)
        F1_list.extend(F1)
        torch.cuda.empty_cache()
        gc.collect()
    P_w = sum([w*p for w,p in zip(weights,P_list)])/sum(weights)
    R_w = sum([w*r for w,r in zip(weights,R_list)])/sum(weights)
    F1_w = sum([w*f for w,f in zip(weights,F1_list)])/sum(weights)
    return P_w, R_w, F1_w

# ---------- Evaluation function ----------
def evaluate_text(gen_text, personality):
    gen_clean = normalize(gen_text)
    ref_texts = df[df["personality"]==personality]["output"].tolist()
    if not ref_texts:
        return None

    ref_clean = [normalize(r) for r in ref_texts]

    # SBERT similarity weights
    gen_emb = sbert_model.encode([gen_clean], convert_to_tensor=True)
    ref_embs = encode_batch(ref_clean, sbert_model)
    sim_scores = util.cos_sim(gen_emb, ref_embs)[0].cpu().numpy()
    best_idx = sim_scores.argmax()
    best_ref = ref_clean[best_idx]
    weights = sim_scores / sim_scores.sum()

    # Weighted BERTScore using best semantic reference
    P_w, R_w, F1_w = bert_score_weighted(gen_clean, [best_ref], [1.0])

    # SBERT semantic similarity
    sem_sim = float(sim_scores.max())

    # BLEU & ROUGE-L
    bleu = sentence_bleu([best_ref.split()], gen_clean.split(), smoothing_function=SmoothingFunction().method1)
    rougeL = rouge.score(best_ref, gen_clean)['rougeL'].fmeasure

    return {
        "BLEU": bleu,
        "ROUGE_L": rougeL,
        "BERT_P": P_w.item(),
        "BERT_R": R_w.item(),
        "BERT_F1": F1_w.item(),
        "SBERT_Sim": sem_sim,
        "Best_Ref": best_ref
    }

# ---------- Find optimal checkpoints ----------
OUTPUT_DIR = "/content/drive/MyDrive/LoRA_outputs_final"
# Use the last few checkpoints (assume lower training loss later in training)
checkpoints = sorted(glob(f"{OUTPUT_DIR}/checkpoint-*"), key=os.path.getmtime)[-5:]

results_table = []

test_prompt = f"""
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Design a space that represents the user’s personality and nurtures their emotional needs.

### Input:
Personality: INFP
Primary Design Inspiration:
 Style: modern | Palette: darkgray, darkslategrey, saddlebrown, sienna, dimgray | Description: a living room with couch and television

Related Inspirations:
1.  Style: modern | Palette: darkgray, darkslategrey, saddlebrown, sienna, dimgray | Description: a living room with television and couch
2.  Style: modern | Palette: darkgray, black, sienna, saddlebrown, dimgray | Description: a living room with couch and television
3.  Style: modern | Palette: darkgray, darkslategrey, sienna, saddlebrown, dimgray | Description: a living room with couch, chair, and television

### Response:
"""

# ---------- Generate & Evaluate ----------
for ckpt in checkpoints:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=ckpt,
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
        device_map="auto"
    )
    model = FastLanguageModel.for_inference(model)
    model.eval()

    inputs = tokenizer([test_prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=500, temperature=0.85, top_p=0.92, do_sample=True, repetition_penalty=1.15)
    gen_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    eval_res = evaluate_text(gen_text, personality="INFP")
    results_table.append({"Checkpoint": os.path.basename(ckpt), "Generated_Text": gen_text, **eval_res})

# ---------- Show results ----------
df_results = pd.DataFrame(results_table)
df_results = df_results.sort_values(by="BERT_F1", ascending=False).reset_index(drop=True)
print("\n===== Recommendation Table (Best on top) =====")
display(df_results)


# Custom Dataset Creation Process: testing


In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree

# --- Your strict style rules ---
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

# --- Optional: Create personality-style correlation matrix ---
def create_correlation_matrix():
    df_rules = pd.DataFrame([{'personality_type': pt, 'styles': styles} for pt, styles in STYLE_RULES.items()])
    mlb = MultiLabelBinarizer()
    style_encoded = mlb.fit_transform(df_rules['styles'])
    corr_matrix = pd.DataFrame(cosine_similarity(style_encoded),
                               columns=df_rules['personality_type'],
                               index=df_rules['personality_type'])
    return corr_matrix

# --- Fetch matching designs for a personality ---
def fetch_matching_designs(personality_type, df_designs):
    if personality_type not in STYLE_RULES:
        raise ValueError(f"Invalid personality type. Must be one of: {list(STYLE_RULES.keys())}")
    target_styles = STYLE_RULES[personality_type]
    mask = df_designs['predicted_style'].apply(lambda x: any(style.lower() in x.lower() for style in target_styles))
    return df_designs[mask].copy()

# --- Palette preprocessing: RGB tuple -> nearest CSS4 color ---
names = list(mcolors.CSS4_COLORS.keys())
rgb_values = [mcolors.to_rgb(mcolors.CSS4_COLORS[name]) for name in names]
kdtree = KDTree(rgb_values)

def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

def rgb_to_color_name(rgb_tuple):
    try:
        # Normalize RGB to 0-1
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

def palette_to_names(palette_str):
    return ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(palette_str)])

# --- Create intent table for all personality types ---
def create_intent_table(df_designs):
    intent_rows = []

    for personality in STYLE_RULES:
        matching_designs = fetch_matching_designs(personality, df_designs)
        # Deduplicate based on style, palette, and description
        matching_designs = matching_designs.drop_duplicates(subset=['predicted_style', 'palette', 'cleaned_description'])

        # Build intent row
        for _, row in matching_designs.iterrows():
            intent_rows.append({
                'personality': personality,
                'style': row['predicted_style'],
                'color_pref': row['palette'],
                'palette_names': palette_to_names(row['palette']),
                'design_motif': row['cleaned_description']
            })

    intent_df = pd.DataFrame(intent_rows)
    intent_df.to_csv("intent_table_all_personalities_preprocessed.csv", index=False)
    print(f"Intent table created with {len(intent_df)} rows!")
    return intent_df

# --- Example usage ---
if __name__ == "__main__":
    # Load your dataset
    # df_with_ROS should have columns: folder_name, image_name, dominant_color, palette, description, cleaned_description, predicted_style
    # replace with your file

    # Optional: show correlation matrix
    corr_matrix = create_correlation_matrix()
    print("Personality Style Correlation Matrix:")
    print(corr_matrix)

    # Create intent table for all personality types with palette preprocessing
    intent_df = create_intent_table(df_with_ROS)
    print(intent_df.head())


In [ ]:
print("Intent table")

intent_df.shape
intent_df.describe()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import json
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree

# ------------------------------
# 1️⃣ Load the dataset
# ------------------------------
intent_df = pd.read_csv("intent_table_all_personalities_preprocessed.csv")
print(f"Loaded dataset: {len(intent_df)} samples")

# ------------------------------
# 2️⃣ Palette preprocessing
# ------------------------------
names = list(mcolors.CSS4_COLORS.keys())
rgb_values = [mcolors.to_rgb(mcolors.CSS4_COLORS[name]) for name in names]
kdtree = KDTree(rgb_values)

def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

def palette_to_names(palette_str):
    return ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(palette_str)])

if 'palette_names' not in intent_df.columns:
    intent_df['palette_names'] = intent_df['color_pref'].apply(palette_to_names)

# ------------------------------
# 3️⃣ Unified text representation
# ------------------------------
intent_df['text_repr'] = (
    intent_df['personality'].astype(str) + " " +
    intent_df['style'].astype(str) + " " +
    intent_df['palette_names'].astype(str) + " " +
    intent_df['design_motif'].astype(str)
)

# ------------------------------
# 4️⃣ Global TF-IDF for consistent feature space
# ------------------------------
print("\nVectorizing all samples...")
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(intent_df['text_repr'])
print("Feature matrix shape:", X.shape)

# ------------------------------
# 5️⃣ Compute global cosine similarity matrix
# ------------------------------
print("Computing global cosine similarity matrix (for correlation awareness)...")
global_similarity = cosine_similarity(X)
print("Global cosine matrix shape:", global_similarity.shape)

# ------------------------------
# 6️⃣ Select diverse samples per personality + style
# ------------------------------
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------
# 5️⃣–6️⃣ Compute cosine similarity per group (not global)
# ------------------------------
print("\nComputing cosine similarity group-wise (optimized for Colab)...")

def select_low_corr_samples_texts(texts, n_select=3):
    """
    Compute cosine similarity for small batch of text samples
    and return indices of least correlated ones.
    """
    if len(texts) <= n_select:
        return list(range(len(texts)))

    X_group = vectorizer.transform(texts)
    sim_matrix = cosine_similarity(X_group)
    mean_corr = sim_matrix.mean(axis=1)
    selected = np.argsort(mean_corr)[:n_select]
    return selected

final_rows = []

for personality in tqdm(intent_df['personality'].unique(), desc="Selecting diverse samples"):
    p_group = intent_df[intent_df['personality'] == personality]
    styles = p_group['style'].unique()

    for style in styles:
        s_group = p_group[p_group['style'] == style]
        if len(s_group) == 0:
            continue

        selected_idx = select_low_corr_samples_texts(s_group['text_repr'].tolist(), n_select=3)
        final_rows.append(s_group.iloc[selected_idx])

final_df = pd.concat(final_rows).reset_index(drop=True)
print(f"\n✅ Final selected dataset size: {len(final_df)}")


# ------------------------------
# 7️⃣ Correlation summary
# ------------------------------
avg_corr = np.mean(global_similarity)
print(f"Average overall dataset correlation: {avg_corr:.4f}")

# ------------------------------
# 8️⃣ Save LoRA-ready JSONL
# ------------------------------
def convert_to_jsonl(df, filename="lora_finetune_dataset_cosine_selected.jsonl"):
    with open(filename, "w") as f:
        for _, row in df.iterrows():
            record = {
                "instruction": f"Recommend an interior design for a {row['personality']} personality.",
                "input": "",
                "output": f"Style: {row['style']}. Palette: {row['palette_names']}. Motif: {row['design_motif']}. This design suits the {row['personality']} personality."
            }
            f.write(json.dumps(record) + "\n")
    print(f"LoRA-ready dataset saved → {filename}")

convert_to_jsonl(final_df)


# Dynamic Cluster Pipeline

In [ ]:
# 🎨 Step 3: Convert Palette Colors to Readable Names
import ast
import matplotlib.colors as mcolors
from scipy.spatial import KDTree
import numpy as np
css3_db = mcolors.CSS4_COLORS
names = list(css3_db.keys())
rgb_values = [mcolors.to_rgb(css3_db[name]) for name in names]
kdtree = KDTree(rgb_values)

def parse_palette(palette_str):
    try:
        return ast.literal_eval(palette_str)
    except:
        return []

def rgb_to_color_name(rgb_tuple):
    try:
        norm_rgb = tuple(x / 255.0 for x in rgb_tuple)
        _, idx = kdtree.query(norm_rgb)
        return names[idx]
    except:
        return "unknown"

df_with_ROS['palette_names'] = df_with_ROS['palette'].apply(
    lambda x: ', '.join([rgb_to_color_name(rgb) for rgb in parse_palette(x)])
)

print("🎨 Added palette_names column:")
print(df_with_ROS[['palette', 'palette_names']].head())


# Cluster-based Mapping on df_with_ROS

In [ ]:
# ===== BLOCK A: Cluster-based Mapping on df_with_ROS =====
import pandas as pd
import json
import random
from pathlib import Path
import ast
from collections import defaultdict

# --- Output Directory ---
OUT_DIR = Path("cluster_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# --- Style & Cluster Definitions ---
STYLE_RULES = {
    'ISTP': ['Vintage', 'Modern'],
    'ISTJ': ['Modern', 'Bohemian', 'Vintage'],
    'ISFP': ['Vintage'],
    'ISFJ': ['Minimalist', 'Vintage', 'Bohemian'],
    'INTP': ['Modern', 'Industrial', 'Vintage'],
    'INTJ': ['Vintage', 'Industrial', 'Bohemian', 'Modern'],
    'INFP': ['Modern', 'Vintage', 'Minimalist'],
    'INFJ': ['Industrial', 'Vintage', 'Modern'],
    'ESTP': ['Modern', 'Vintage'],
    'ESTJ': ['Modern', 'Vintage', 'Bohemian'],
    'ESFP': ['Bohemian', 'Modern'],
    'ESFJ': ['Vintage'],
    'ENTP': ['Modern', 'Vintage'],
    'ENTJ': ['Vintage', 'Bohemian', 'Modern'],
    'ENFP': ['Vintage', 'Bohemian', 'Industrial'],
    'ENFJ': ['Modern', 'Vintage']
}

CLUSTERS = {
    'DIT': ['INFJ', 'INTJ', 'ENFP', 'ENTP'],
    'DST': ['ISTP', 'ISFP', 'ESTP', 'ESFP'],
    'DTT': ['ISTJ', 'INTJ', 'ESTJ', 'ENTJ'],
    'DFT': ['ISFJ', 'INFJ', 'ESFJ', 'ENFJ']
}

random.seed(42)

# --- Step 0: Ensure df_with_ROS exists ---
if 'df_with_ROS' not in globals():
    raise RuntimeError("❌ df_with_ROS not found in memory. Please load it first.")

df = df_with_ROS.copy()

# Accept minor variations in column naming
if 'cleaned_description' not in df.columns and 'description' in df.columns:
    df = df.rename(columns={'description': 'cleaned_description'})

for col in ['predicted_style', 'cleaned_description', 'palette']:
    if col not in df.columns:
        df[col] = ""

print("✅ Dataset loaded successfully. Shape:", df.shape)
display(df.head(3))

# --- Step 1: Build Personality–Style Matrix ---
def build_style_matrix(style_rules):
    all_styles = sorted({s for v in style_rules.values() for s in v})
    types = sorted(style_rules.keys())
    M = pd.DataFrame(0, index=types, columns=all_styles, dtype=int)
    for t, styles in style_rules.items():
        for s in styles:
            M.loc[t, s] = 1
    print("\n🧩 Style–Personality Matrix (preview):")
    display(M.head())
    return M

# --- Step 2: Compute Representative Styles per Cluster ---
def compute_cluster_style_map(style_rules, clusters, threshold=0.5, fallback_topk=2):
    M = build_style_matrix(style_rules)
    cluster_map = {}

    print("\n📊 Computing style prominence within each cluster...")
    for cname, members in clusters.items():
        valid_members = [m for m in members if m in M.index]
        S = M.loc[valid_members].sum(axis=0)           # style frequency
        P = (S / len(valid_members)).round(3)          # prominence ratio
        prominents = list(P[P >= threshold].index)
        if not prominents:
            prominents = list(P.sort_values(ascending=False).head(fallback_topk).index)
        cluster_map[cname] = {
            "members": valid_members,
            "representative_styles": prominents,
            "prominence": P.to_dict()
        }

        print(f"\n🔹 Cluster {cname}")
        print("   Members:", valid_members)
        print("   Representative Styles:", prominents)
        display(P.sort_values(ascending=False))

    return cluster_map

cluster_map = compute_cluster_style_map(STYLE_RULES, CLUSTERS)

# --- Step 3: Map Dataset Rows to Clusters Based on Representative Styles ---
def map_rows_to_clusters(df, cluster_map, out_dir=OUT_DIR):
    for cname, info in cluster_map.items():
        reps = info['representative_styles']
        members = info['members']

        # Filter dataset rows whose predicted style matches any representative style
        mask = df['predicted_style'].str.lower().apply(lambda x: any(r.lower() in x for r in reps))
        subset = df[mask].copy()

        # Add mapping metadata
        subset['cluster_name'] = cname
        subset['mapped_personalities'] = ", ".join(members)

        # Save feature-only CSV
        csv_path = out_dir / f"{cname}_cluster_raw.csv"
        subset.to_csv(csv_path, index=False)

        print(f"\n✅ Saved {cname}_cluster_raw.csv | Rows: {len(subset)}")
        display(subset.head(3))

map_rows_to_clusters(df, cluster_map)

print("\n🎯 BLOCK A complete — Feature-based cluster CSVs generated successfully!")
print("📁 Files saved in:", OUT_DIR.resolve())


In [ ]:
# --- Step 4: Cluster Statistics Summary ---
import os

summary = []

for cname in CLUSTERS.keys():
    file_path = OUT_DIR / f"{cname}_cluster_raw.csv"
    if file_path.exists():
        temp_df = pd.read_csv(file_path)
        num_rows, num_cols = temp_df.shape
        summary.append({
            "Cluster": cname,
            "Rows": num_rows,
            "Columns": num_cols,
            "Representative_Styles": ", ".join(cluster_map[cname]["representative_styles"]),
            "Mapped_Personalities": ", ".join(cluster_map[cname]["members"])
        })
    else:
        summary.append({
            "Cluster": cname,
            "Rows": 0,
            "Columns": 0,
            "Representative_Styles": "—",
            "Mapped_Personalities": "—"
        })

summary_df = pd.DataFrame(summary)
print("\n📊 Cluster Summary Report:")
display(summary_df)

print("\n📈 Overall dataset size:", df.shape)
print("📦 Total records across clusters:", summary_df['Rows'].sum())


# Cluster–Style Correlation Matrix

In [ ]:
# ===== BLOCK B: Cluster–Style Correlation Matrix =====
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ensure cluster_map exists (from Block A)
if 'cluster_map' not in globals():
    raise RuntimeError("❌ cluster_map not found. Run Block A first.")

OUT_DIR = Path("cluster_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

# --- Build full correlation matrix ---
all_styles = sorted({s for v in STYLE_RULES.values() for s in v})
clusters = sorted(cluster_map.keys())

corr_matrix = pd.DataFrame(0.0, index=all_styles, columns=clusters)

for cname, info in cluster_map.items():
    for style, ratio in info['prominence'].items():
        corr_matrix.loc[style, cname] = ratio

print("\n📊 Cluster–Style Correlation Matrix:")
display(corr_matrix.round(3))

# --- Save overall matrix ---
corr_path = OUT_DIR / "cluster_style_correlation_matrix.csv"
corr_matrix.to_csv(corr_path)
print(f"✅ Saved overall correlation matrix → {corr_path}")

# --- Generate per-cluster breakdowns ---
for cname in clusters:
    sub_df = corr_matrix[[cname]].copy()
    sub_df.columns = ["Prominence_Ratio"]
    sub_df["Cluster"] = cname
    sub_path = OUT_DIR / f"{cname}_style_correlation.csv"
    sub_df.to_csv(sub_path)
    print(f"✅ Saved {cname} style correlation → {sub_path}")
    display(sub_df.sort_values("Prominence_Ratio", ascending=False).head(5))

# --- Optional Visualization ---
plt.figure(figsize=(8, 5))
sns.heatmap(corr_matrix, annot=True, cmap="YlGnBu", fmt=".2f", cbar_kws={'label': 'Prominence Ratio'})
plt.title("Cluster–Style Correlation Heatmap", fontsize=14, pad=15)
plt.xlabel("Clusters")
plt.ylabel("Styles")
plt.tight_layout()

plt_path = OUT_DIR / "cluster_style_correlation_heatmap.png"
plt.savefig(plt_path, dpi=300)
plt.show()
print(f"📈 Heatmap saved → {plt_path}")

print("\n🎯 BLOCK B complete — Correlation matrices + heatmap generated!")


In [ ]:
!jupyter nbconvert --to html Home_Fit_Final.ipynb